In [0]:
import pandas as pd
import numpy as np
import os, json, calendar
from datetime import datetime, date
from dateutil.relativedelta import relativedelta
from pyspark.sql import functions as F

In [0]:
# ═══════════════════════════════════════════════════════════════════════
# Widgets
# ═══════════════════════════════════════════════════════════════════════

dbutils.widgets.dropdown(
    "target_quarter", "2026-Q2",
    [f"{y}-Q{q}" for y in range(2024, 2028) for q in range(1, 5)],
    "Target Quarter",
)

dbutils.widgets.text("client_csv_filename", "proposed_csm_allocation.csv", "Client CSV Filename")
dbutils.widgets.text("take_rate_override_pct", "", "Take Rate Override (%)")
dbutils.widgets.text("growth_rate_override_pct", "", "Growth Rate Override (%)")

print("Widgets ready.")


In [0]:
# ═══════════════════════════════════════════════════════════════════════
# Utilities — constants, helpers, formatters
# ═══════════════════════════════════════════════════════════════════════

QUARTER_MONTHS = {1: (1, 3), 2: (4, 6), 3: (7, 9), 4: (10, 12)}

PALETTE = [
    "#4A90E2", "#50A684", "#F5A623", "#D0021B",
    "#8E44AD", "#34495E", "#16A085", "#2C3E50",
    "#E67E22", "#27AE60", "#2980B9", "#C0392B",
]

BG_COLORS = {"SG": "#4A90E2", "HK": "#D0021B", "AU": "#50A684", "ID": "#F5A623"}
BU_COLORS = {"AspireBA": "#4A90E2", "AspireOS": "#50A684", "AspireX": "#F5A623"}
MOTION_COLORS = {"Key": "#8E44AD", "Scaled": "#16A085"}

def format_usd(value):
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return "$0.00"
    abs_val = abs(value)
    if abs_val >= 1_000_000_000:
        return f"${value / 1_000_000_000:,.2f}B"
    if abs_val >= 1_000_000:
        return f"${value / 1_000_000:,.2f}M"
    if abs_val >= 1_000:
        return f"${value / 1_000:,.2f}K"
    return f"${value:,.2f}"

def fmt_pct(v):
    return f"{v * 100:.2f}%" if v is not None and not (isinstance(v, float) and pd.isna(v)) else "0.00%"

def quarter_to_dates(q_str):
    year, q = int(q_str[:4]), int(q_str[-1])
    ms, me = QUARTER_MONTHS[q]
    return date(year, ms, 1), date(year, me, calendar.monthrange(year, me)[1])

def date_to_quarter(d):
    return f"{d.year}-Q{(d.month - 1) // 3 + 1}"

def current_quarter():
    return date_to_quarter(date.today())

def quarter_offset(q_str, offset):
    year, q = int(q_str[:4]), int(q_str[-1])
    total = (year * 4 + q - 1) + offset
    ny, nq = divmod(total, 4)
    return f"{ny}-Q{nq + 1}"

def q_to_int(q_str):
    y, q = int(q_str[:4]), int(q_str[-1])
    return y * 4 + q

def quarters_between(qs, qe):
    result, cur = [], qs
    while cur <= qe:
        result.append(cur)
        cur = quarter_offset(cur, 1)
    return result

TARGET_QUARTER  = dbutils.widgets.get("target_quarter")
CSV_FILENAME    = dbutils.widgets.get("client_csv_filename")
CURRENT_QUARTER = current_quarter()

# Resolve notebook folder path
try:
    _nb_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
    NB_DIR = "/".join(_nb_path.rsplit("/", 1)[:-1])
except Exception:
    NB_DIR = "."

CSV_PATH = f"/Workspace{NB_DIR}/{CSV_FILENAME}"

print(f"Current Qtr : {CURRENT_QUARTER}")
print(f"Target Qtr  : {TARGET_QUARTER}")
print(f"CSV Path    : {CSV_PATH}")


In [0]:
# ═══════════════════════════════════════════════════════════════════════
# Load Client CSV (same folder as notebook)
# ═══════════════════════════════════════════════════════════════════════

EXPECTED_COLS = [
    "business_id", "business_ref_code", "company_name",
    "business_group", "business_unit", "account_motion", "account_manager",
]

client_df = None

# Method 1: Pandas from Workspace path (bypasses UC file SELECT restriction)
try:
    client_pd = pd.read_csv(CSV_PATH)
    client_df = spark.createDataFrame(client_pd)
    print(f"Loaded via pd.read_csv: {CSV_PATH}")
except Exception as e1:
    print(f"Method 1 failed: {e1}")

# Method 2: spark file: protocol
if client_df is None:
    try:
        client_df = spark.read.option("header", "true").option("inferSchema", "true").csv(f"file:{CSV_PATH}")
        print(f"Loaded via spark file: protocol")
    except Exception as e2:
        print(f"Method 2 failed: {e2}")

if client_df is None:
    raise RuntimeError(f"Cannot load {CSV_FILENAME}. Ensure it's in the same repo folder as this notebook.")

# Normalise columns
for c in client_df.columns:
    client_df = client_df.withColumnRenamed(c, c.strip().lower().replace(" ", "_"))
missing = [c for c in EXPECTED_COLS if c not in client_df.columns]
if missing:
    raise ValueError(f"Missing columns: {missing}.  Found: {client_df.columns}")

client_df = client_df.select(EXPECTED_COLS)
clients_pd = client_df.toPandas()
clients_pd["business_id"] = clients_pd["business_id"].astype(str)

print(f"Clients loaded: {len(clients_pd)}")


In [0]:
# ═══════════════════════════════════════════════════════════════════════
# Revenue Data — global session cache
# ═══════════════════════════════════════════════════════════════════════

if "_REV_CACHE" not in dir():
    _client_ids = ",".join([f"'{x}'" for x in clients_pd["business_id"].unique()])
    _, _q_end = quarter_to_dates(CURRENT_QUARTER)
    _q_end_str = _q_end.strftime("%Y-%m-%d")

    _REV_CACHE = spark.sql(f"""
        SELECT
            CAST(r.business_id AS STRING) AS business_id,
            CAST(r.transaction_date AS DATE) AS txn_date,
            SUM(r.gross_revenue)     AS gross_revenue,
            SUM(r.gross_profit)      AS gross_profit
        FROM cat_legacy.analytics.revenue_core r
        WHERE r.category <> 'Credit'
          AND r.sub_product <> 'Incremental Float'
          AND r.gross_revenue IS NOT NULL
          AND CAST(r.business_id AS STRING) IN ({_client_ids})
          AND r.transaction_date BETWEEN '2025-01-01' AND '{_q_end_str}'
        GROUP BY 1, 2

        UNION ALL

        SELECT
            fv.business_id,
            fv.date AS txn_date,
            SUM(fv.csm_core_float_revenue) AS gross_revenue,
            SUM(fv.csm_core_float_revenue) AS gross_profit
        FROM cat_legacy.analytics.float_view fv
        WHERE fv.date BETWEEN '2025-01-01' AND '{_q_end_str}'
          AND CAST(fv.business_id AS STRING) IN ({_client_ids})
        GROUP BY 1, 2
    """).toPandas()

    _REV_CACHE["txn_date"] = pd.to_datetime(_REV_CACHE["txn_date"])
    _REV_CACHE["year"]     = _REV_CACHE["txn_date"].dt.year
    _REV_CACHE["month"]    = _REV_CACHE["txn_date"].dt.month
    _REV_CACHE["quarter"]  = _REV_CACHE.apply(lambda r: f"{r['year']}-Q{(r['month']-1)//3+1}", axis=1)

    print(f"Revenue cache: {len(_REV_CACHE):,} rows")
else:
    print("Revenue cache already loaded — skipped re-fetch")

raw_rev = _REV_CACHE


In [0]:
# ═══════════════════════════════════════════════════════════════════════
# Quarterly Aggregations — Client level (enriched with client metadata)
# ═══════════════════════════════════════════════════════════════════════

# Merge revenue with client metadata
rev = raw_rev.merge(clients_pd, on="business_id", how="inner")

# Client-Quarter level
q_client = (
    rev.groupby(["business_id", "business_ref_code", "company_name",
                 "business_group", "business_unit", "account_motion",
                 "account_manager", "quarter"])
    .agg(gross_revenue=("gross_revenue", "sum"),
         gross_profit=("gross_profit", "sum"),
         active_days=("txn_date", "nunique"))
    .reset_index()
)
q_client["take_rate"] = np.where(
    q_client["gross_revenue"] != 0,
    q_client["gross_profit"] / q_client["gross_revenue"], 0
)

# Overall quarterly
q_overall = (
    rev.groupby("quarter")
    .agg(gross_revenue=("gross_revenue", "sum"),
         gross_profit=("gross_profit", "sum"))
    .reset_index()
    .sort_values("quarter")
)
q_overall["take_rate"] = np.where(
    q_overall["gross_revenue"] != 0,
    q_overall["gross_profit"] / q_overall["gross_revenue"], 0
)

print(f"Quarterly client rows : {len(q_client):,}")
print(f"Quarters available    : {sorted(q_client['quarter'].unique())}")


In [0]:
# ═══════════════════════════════════════════════════════════════════════
# Time Logic Engine — resolve L4Q, estimate current quarter if needed
# ═══════════════════════════════════════════════════════════════════════

def estimate_current_quarter_client(rev_df, clients_pd, current_q):
    """Estimate current (incomplete) quarter: actuals + daily run-rate projection."""
    today = date.today()
    q_start, q_end = quarter_to_dates(current_q)

    cq_data = rev_df[rev_df["quarter"] == current_q].copy()
    if cq_data.empty:
        return None

    current_month = today.month
    days_elapsed = today.day
    total_days_month = calendar.monthrange(today.year, current_month)[1]

    completed_months = [m for m in range(q_start.month, current_month)]
    future_months = [m for m in range(current_month + 1, q_end.month + 1)]

    # Completed months actuals
    completed = cq_data[cq_data["month"].isin(completed_months)].groupby("business_id").agg(
        rev_completed=("gross_revenue", "sum"), gp_completed=("gross_profit", "sum")
    ).reset_index() if completed_months else pd.DataFrame(columns=["business_id", "rev_completed", "gp_completed"])

    # Ongoing month run-rate
    ongoing = cq_data[cq_data["month"] == current_month].groupby("business_id").agg(
        rev_ongoing=("gross_revenue", "sum"), gp_ongoing=("gross_profit", "sum")
    ).reset_index()
    ongoing["rev_proj"] = ongoing["rev_ongoing"] / days_elapsed * total_days_month
    ongoing["gp_proj"]  = ongoing["gp_ongoing"]  / days_elapsed * total_days_month

    n_future = len(future_months)

    # Combine
    est = clients_pd[["business_id"]].copy()
    est = est.merge(completed, on="business_id", how="left").fillna(0)
    est = est.merge(ongoing[["business_id", "rev_proj", "gp_proj"]], on="business_id", how="left").fillna(0)
    est["gross_revenue"] = est["rev_completed"] + est["rev_proj"] + est["rev_proj"] * n_future
    est["gross_profit"]  = est["gp_completed"]  + est["gp_proj"]  + est["gp_proj"]  * n_future
    est["quarter"] = current_q
    est["take_rate"] = np.where(est["gross_revenue"] != 0, est["gross_profit"] / est["gross_revenue"], 0)
    return est[["business_id", "quarter", "gross_revenue", "gross_profit", "take_rate"]]


def resolve_l4q(target_q, current_q, q_client_df, rev_df, clients_pd_df):
    target_int  = q_to_int(target_q)
    current_int = q_to_int(current_q)
    available   = sorted(q_client_df["quarter"].unique())

    # CASE 1: past quarter
    if target_int <= current_int:
        candidates = [q for q in available if q_to_int(q) < target_int]
        l4q = candidates[-4:] if len(candidates) >= 4 else candidates
        return l4q, q_client_df

    # CASE 2: next immediate quarter
    if target_int == current_int + 1:
        est = estimate_current_quarter_client(rev_df, clients_pd_df, current_q)
        if est is not None:
            base = q_client_df[q_client_df["quarter"] != current_q].copy()
            est_full = est.merge(clients_pd_df, on="business_id", how="left")
            est_full["active_days"] = 0
            combined = pd.concat([base, est_full], ignore_index=True)
            past = [q for q in available if q_to_int(q) < current_int]
            l4q = (past[-3:] + [current_q]) if len(past) >= 3 else (past + [current_q])
        else:
            combined = q_client_df
            candidates = [q for q in available if q_to_int(q) < target_int]
            l4q = candidates[-4:]
        return l4q, combined

    # CASE 3: far future — iterative projection
    est = estimate_current_quarter_client(rev_df, clients_pd_df, current_q)
    if est is not None:
        base = q_client_df[q_client_df["quarter"] != current_q].copy()
        est_full = est.merge(clients_pd_df, on="business_id", how="left")
        est_full["active_days"] = 0
        combined = pd.concat([base, est_full], ignore_index=True)
    else:
        combined = q_client_df.copy()

    all_q = sorted(combined["quarter"].unique())
    last_known = all_q[-1] if all_q else quarter_offset(current_q, -1)
    intermediates = quarters_between(quarter_offset(last_known, 1), quarter_offset(target_q, -1))

    for proj_q in intermediates:
        avail_q = sorted(combined["quarter"].unique())
        proj_l4q = avail_q[-4:] if len(avail_q) >= 4 else avail_q
        l4q_sub = combined[combined["quarter"].isin(proj_l4q)]

        growth = _compute_client_growth(l4q_sub)
        last_q_data = combined[combined["quarter"] == avail_q[-1]].copy()
        last_q_data = last_q_data.merge(growth[["business_id", "avg_rev_g"]], on="business_id", how="left").fillna(0)
        last_q_data["gross_revenue"] = last_q_data["gross_revenue"] * (1 + last_q_data["avg_rev_g"])
        last_q_data["gross_profit"]  = last_q_data["gross_profit"]  * (1 + last_q_data["avg_rev_g"])
        last_q_data["quarter"] = proj_q
        last_q_data["take_rate"] = np.where(last_q_data["gross_revenue"] != 0, last_q_data["gross_profit"] / last_q_data["gross_revenue"], 0)
        last_q_data.drop(columns=["avg_rev_g"], inplace=True, errors="ignore")
        combined = pd.concat([combined, last_q_data], ignore_index=True)

    final_avail = sorted(combined["quarter"].unique())
    candidates = [q for q in final_avail if q_to_int(q) < target_int]
    l4q = candidates[-4:] if len(candidates) >= 4 else candidates
    return l4q, combined


print("Time Logic Engine ready")


In [0]:
# ═══════════════════════════════════════════════════════════════════════
# Portfolio Growth Engine — Weighted QoQ + Seasonal Index + Momentum
# ═══════════════════════════════════════════════════════════════════════

# _compute_client_growth is still needed by resolve_l4q for Case-3 projection
def _compute_client_growth(q_df):
    """Compute simple average QoQ growth per client (used by resolve_l4q)."""
    q_df = q_df.sort_values(["business_id", "quarter"])
    q_df["prev_rev"] = q_df.groupby("business_id")["gross_revenue"].shift(1)
    q_df["prev_gp"]  = q_df.groupby("business_id")["gross_profit"].shift(1)
    q_df["rev_g"] = np.where((q_df["prev_rev"].notna()) & (q_df["prev_rev"].abs() > 0),
                             (q_df["gross_revenue"] - q_df["prev_rev"]) / q_df["prev_rev"].abs(), np.nan)
    q_df["gp_g"]  = np.where((q_df["prev_gp"].notna()) & (q_df["prev_gp"].abs() > 0),
                             (q_df["gross_profit"] - q_df["prev_gp"]) / q_df["prev_gp"].abs(), np.nan)
    growth = q_df.groupby("business_id").agg(avg_rev_g=("rev_g", "mean"), avg_gp_g=("gp_g", "mean"), periods=("rev_g", "count")).reset_index()
    return growth

# ── Execute ───────────────────────────────────────────────────────────
l4q_quarters, enriched_qc = resolve_l4q(TARGET_QUARTER, CURRENT_QUARTER, q_client, rev, clients_pd)
print(f"L4Q: {l4q_quarters}")

l4q_data = enriched_qc[enriched_qc["quarter"].isin(l4q_quarters)]

# ── 1. Portfolio QoQ Growth Rates ─────────────────────────────────────
port_q = (
    l4q_data.groupby("quarter")
    .agg(port_rev=("gross_revenue", "sum"), port_gp=("gross_profit", "sum"))
    .reset_index()
    .sort_values("quarter")
)
port_q["prev_rev"] = port_q["port_rev"].shift(1)
port_q["prev_gp"]  = port_q["port_gp"].shift(1)
port_q["rev_g"] = np.where(
    (port_q["prev_rev"].notna()) & (port_q["prev_rev"].abs() > 0),
    (port_q["port_rev"] - port_q["prev_rev"]) / port_q["prev_rev"].abs(), np.nan)
port_q["gp_g"] = np.where(
    (port_q["prev_gp"].notna()) & (port_q["prev_gp"].abs() > 0),
    (port_q["port_gp"] - port_q["prev_gp"]) / port_q["prev_gp"].abs(), np.nan)

rev_growths = port_q["rev_g"].dropna().values
gp_growths  = port_q["gp_g"].dropna().values
n_g = len(rev_growths)

# ── 2. Recency-Weighted QoQ Growth ───────────────────────────────────
# Exponential weights: most recent quarter gets highest weight
if n_g > 0:
    raw_w = np.array([2 ** i for i in range(n_g)], dtype=float)
    w = raw_w / raw_w.sum()
    weighted_rev_g = float(np.dot(w, rev_growths))
    weighted_gp_g  = float(np.dot(w, gp_growths)) if len(gp_growths) == n_g else weighted_rev_g
    simple_rev_g   = float(np.mean(rev_growths))
else:
    weighted_rev_g = simple_rev_g = 0.0
    weighted_gp_g  = 0.0

# ── 3. Momentum Detection ────────────────────────────────────────────
# Is growth accelerating or decelerating?
if n_g >= 3:
    recent_half = rev_growths[n_g // 2:]
    older_half  = rev_growths[:n_g // 2]
    momentum = float(np.mean(recent_half) - np.mean(older_half))
    momentum_label = "Accelerating" if momentum > 0.005 else ("Decelerating" if momentum < -0.005 else "Stable")
elif n_g >= 2:
    momentum = float(rev_growths[-1] - rev_growths[0])
    momentum_label = "Accelerating" if momentum > 0.005 else ("Decelerating" if momentum < -0.005 else "Stable")
else:
    momentum = 0.0
    momentum_label = "Insufficient data"

# ── 4. Seasonal Index ────────────────────────────────────────────────
# If same quarter from prior year exists, compute seasonal adjustment
target_q_num = int(TARGET_QUARTER[-1])  # e.g. 2 for Q2
all_q_data = enriched_qc.groupby("quarter").agg(rev=("gross_revenue", "sum")).reset_index()
all_q_data["q_num"] = all_q_data["quarter"].str[-1].astype(int)

same_q_prior = all_q_data[
    (all_q_data["q_num"] == target_q_num) &
    (all_q_data["quarter"] < TARGET_QUARTER)
].sort_values("quarter")

if len(same_q_prior) > 0 and len(all_q_data) >= 4:
    # Seasonal index = avg revenue in target-quarter-number / avg across all quarters
    avg_same_q_rev = same_q_prior["rev"].mean()
    avg_all_q_rev  = all_q_data["rev"].mean()
    seasonal_index = avg_same_q_rev / avg_all_q_rev if avg_all_q_rev > 0 else 1.0
    seasonal_label = f"{seasonal_index:.3f}"
    HAS_SEASONAL = True
else:
    seasonal_index = 1.0
    seasonal_label = "N/A (insufficient YoY data)"
    HAS_SEASONAL = False

# ── 5. Final Growth Rate ─────────────────────────────────────────────
# Blended: use weighted QoQ as base, apply seasonal adjustment
avg_qoq_growth    = weighted_rev_g
avg_qoq_gp_growth = weighted_gp_g

# ── 6. Growth Rate Override ──────────────────────────────────────────
_gr_override_raw = dbutils.widgets.get("growth_rate_override_pct").strip()
GROWTH_OVERRIDE = len(_gr_override_raw) > 0
if GROWTH_OVERRIDE:
    override_growth = float(_gr_override_raw) / 100.0
    avg_qoq_growth    = override_growth
    avg_qoq_gp_growth = override_growth
    print(f"** GROWTH RATE OVERRIDDEN to {override_growth*100:+.2f}% **")

# ── 7. Target Revenue from Last Quarter ──────────────────────────────
last_q = sorted(l4q_quarters)[-1]
last_q_port = port_q[port_q["quarter"] == last_q]
total_base_rev = last_q_port["port_rev"].values[0]
total_base_gp  = last_q_port["port_gp"].values[0]

# Apply growth + seasonal adjustment
total_target_rev = total_base_rev * (1 + avg_qoq_growth) * seasonal_index
total_target_gp  = total_base_gp  * (1 + avg_qoq_gp_growth) * seasonal_index
target_tr = total_target_gp / total_target_rev if total_target_rev else 0

# ── Historical benchmark Take Rate (L4Q weighted) ────────────────────
l4q_rev_sum = port_q["port_rev"].sum()
l4q_gp_sum  = port_q["port_gp"].sum()
HIST_AVG_TR = l4q_gp_sum / l4q_rev_sum if l4q_rev_sum else 0

# ── Summary ──────────────────────────────────────────────────────────
print(f"Growth method         : Recency-weighted QoQ")
print(f"  Simple avg QoQ      : {simple_rev_g*100:+.2f}%")
print(f"  Weighted QoQ        : {weighted_rev_g*100:+.2f}%")
print(f"  Momentum            : {momentum_label} ({momentum*100:+.2f}pp)")
print(f"  Seasonal index (Q{target_q_num}) : {seasonal_label}")
print(f"  Final growth applied: {avg_qoq_growth*100:+.2f}%")
print(f"Base Rev ({last_q})       : {format_usd(total_base_rev)}")
print(f"Target Rev                : {format_usd(total_target_rev)}")
print(f"Target GP                 : {format_usd(total_target_gp)}")
print(f"Target TR                 : {fmt_pct(target_tr)}")
print(f"Hist Avg TR (L4Q)         : {fmt_pct(HIST_AVG_TR)}")


In [0]:
# ═══════════════════════════════════════════════════════════════════════
# Q-1 Based Target Allocation — Latest-quarter execution capacity
# ═══════════════════════════════════════════════════════════════════════
# Key principle: growth is portfolio-driven, allocation uses Q-1 distribution

# Q-1 data  =  the last quarter in L4Q
q1_data = enriched_qc[enriched_qc["quarter"] == last_q].copy()
q1_total_rev = q1_data["gross_revenue"].sum()

def q1_alloc(df, group_cols, total_target, q1_total):
    """Build allocation from Q-1 revenue distribution."""
    grp = df.groupby(group_cols).agg(
        q1_rev=("gross_revenue", "sum"),
        q1_gp=("gross_profit", "sum"),
    ).reset_index()
    grp["ratio"] = np.where(q1_total > 0, grp["q1_rev"] / q1_total, 0)
    grp["target_rev"] = grp["ratio"] * total_target
    grp["l4q_tr"] = np.where(grp["q1_rev"] != 0, grp["q1_gp"] / grp["q1_rev"], 0)
    return grp.sort_values("target_rev", ascending=False).reset_index(drop=True)

# ── Top-level allocations (portfolio → dimension) ────────────────────
alloc_bg     = q1_alloc(q1_data, ["business_group"],  total_target_rev, q1_total_rev)
alloc_bu     = q1_alloc(q1_data, ["business_unit"],    total_target_rev, q1_total_rev)
alloc_motion = q1_alloc(q1_data, ["account_motion"],   total_target_rev, q1_total_rev)
alloc_csm    = q1_alloc(q1_data, ["account_manager"],  total_target_rev, q1_total_rev)

# ── Cross-tabs ───────────────────────────────────────────────────────
alloc_bu_mot = q1_alloc(q1_data, ["business_unit", "account_motion"], total_target_rev, q1_total_rev)
alloc_bu_mot["avg_contrib"] = alloc_bu_mot["ratio"]  # backward compat with dashboard

# CSM × Motion  — within-CSM split
alloc_csm_mot = q1_alloc(q1_data, ["account_manager", "account_motion"], total_target_rev, q1_total_rev)
alloc_csm_mot["avg_contrib"] = alloc_csm_mot["ratio"]

# ── Within-CSM BU & Motion splits ───────────────────────────────────
# Compute CSM-internal ratios for detailed drill-down
csm_bu_q1 = q1_data.groupby(["account_manager", "business_unit"]).agg(rev=("gross_revenue", "sum")).reset_index()
csm_bu_q1 = csm_bu_q1.merge(
    alloc_csm[["account_manager", "q1_rev"]].rename(columns={"q1_rev": "csm_rev"}),
    on="account_manager")
csm_bu_q1["pct_within_csm"] = np.where(csm_bu_q1["csm_rev"] > 0, csm_bu_q1["rev"] / csm_bu_q1["csm_rev"] * 100, 0)
csm_bu_q1 = csm_bu_q1.merge(
    alloc_csm[["account_manager", "target_rev"]].rename(columns={"target_rev": "csm_target"}),
    on="account_manager")
csm_bu_q1["bu_target"] = csm_bu_q1["pct_within_csm"] / 100 * csm_bu_q1["csm_target"]

csm_mot_q1 = q1_data.groupby(["account_manager", "account_motion"]).agg(rev=("gross_revenue", "sum")).reset_index()
csm_mot_q1 = csm_mot_q1.merge(
    alloc_csm[["account_manager", "q1_rev"]].rename(columns={"q1_rev": "csm_rev"}),
    on="account_manager")
csm_mot_q1["pct_within_csm"] = np.where(csm_mot_q1["csm_rev"] > 0, csm_mot_q1["rev"] / csm_mot_q1["csm_rev"] * 100, 0)
csm_mot_q1 = csm_mot_q1.merge(
    alloc_csm[["account_manager", "target_rev"]].rename(columns={"target_rev": "csm_target"}),
    on="account_manager")
csm_mot_q1["mot_target"] = csm_mot_q1["pct_within_csm"] / 100 * csm_mot_q1["csm_target"]

# ── Client-Level Targets (within-CSM Q-1 share) ─────────────────────
client_q1 = q1_data.merge(
    alloc_csm[["account_manager", "q1_rev", "target_rev"]].rename(
        columns={"q1_rev": "csm_q1_rev", "target_rev": "csm_target_rev"}),
    on="account_manager", how="left"
)
client_q1["ratio_in_csm"] = np.where(
    client_q1["csm_q1_rev"] > 0,
    client_q1["gross_revenue"] / client_q1["csm_q1_rev"], 0)
client_q1["target_revenue"]   = client_q1["ratio_in_csm"] * client_q1["csm_target_rev"]
client_q1["target_take_rate"] = target_tr  # portfolio-level target TR
client_q1["target_gp"]        = client_q1["target_revenue"] * target_tr

# Historical client take rate for guardrails
hist_client_tr = l4q_data.groupby("business_id").agg(
    hist_rev=("gross_revenue", "sum"), hist_gp=("gross_profit", "sum")
).reset_index()
hist_client_tr["hist_avg_tr"] = np.where(
    hist_client_tr["hist_rev"] != 0,
    hist_client_tr["hist_gp"] / hist_client_tr["hist_rev"], 0)
client_q1 = client_q1.merge(hist_client_tr[["business_id", "hist_avg_tr"]], on="business_id", how="left")
client_q1["hist_avg_tr"] = client_q1["hist_avg_tr"].fillna(HIST_AVG_TR)
client_q1["guardrail"] = np.where(client_q1["target_take_rate"] >= client_q1["hist_avg_tr"], "Safe", "Risk")
client_q1["tr_delta"]  = client_q1["target_take_rate"] - client_q1["hist_avg_tr"]

# ── Build targets & alloc_client (backward-compat names) ─────────────
targets = client_q1.copy()
targets.rename(columns={
    "gross_revenue": "base_revenue", "gross_profit": "base_gp",
    "take_rate": "base_take_rate", "quarter": "base_quarter",
}, inplace=True)
targets["target_quarter"] = TARGET_QUARTER
targets["avg_rev_g"] = avg_qoq_growth   # portfolio-level, same for all
targets["avg_gp_g"]  = avg_qoq_gp_growth
targets["gp_trend"]  = np.where(targets["target_gp"] >= targets["base_gp"], "Growing", "Declining")

alloc_client = targets[[
    "business_id", "business_ref_code", "company_name",
    "business_group", "business_unit", "account_motion", "account_manager",
    "base_quarter", "target_quarter",
    "base_revenue", "base_gp", "base_take_rate",
    "avg_rev_g", "avg_gp_g",
    "target_revenue", "target_gp", "target_take_rate",
    "hist_avg_tr", "tr_delta", "guardrail", "gp_trend"
]].sort_values("target_revenue", ascending=False).reset_index(drop=True)

safe_count    = (alloc_client["guardrail"] == "Safe").sum()
risk_count    = (alloc_client["guardrail"] == "Risk").sum()
total_clients = len(alloc_client)

# ── CSM Capacity Metrics ────────────────────────────────────────────
# For each CSM: concentration ratio, required growth, headroom flags

csm_capacity = alloc_csm.copy()

# Client count per CSM
csm_client_counts = q1_data.groupby("account_manager")["business_id"].nunique().reset_index()
csm_client_counts.columns = ["account_manager", "client_count"]
csm_capacity = csm_capacity.merge(csm_client_counts, on="account_manager", how="left")
csm_capacity["client_count"] = csm_capacity["client_count"].fillna(0).astype(int)

# Concentration ratio — top-3 client share of CSM's Q-1 revenue
def top3_concentration(mgr, data):
    csm_data = data[data["account_manager"] == mgr].sort_values("gross_revenue", ascending=False)
    total = csm_data["gross_revenue"].sum()
    if total <= 0 or len(csm_data) == 0:
        return 0.0
    top3 = csm_data.head(3)["gross_revenue"].sum()
    return top3 / total

csm_capacity["top3_concentration"] = csm_capacity["account_manager"].apply(
    lambda m: top3_concentration(m, q1_data)
)

# Required QoQ growth per CSM (target vs Q-1 actual)
csm_capacity["required_growth"] = np.where(
    csm_capacity["q1_rev"] > 0,
    (csm_capacity["target_rev"] - csm_capacity["q1_rev"]) / csm_capacity["q1_rev"],
    0
)

# Average revenue per client (target)
csm_capacity["avg_client_target"] = np.where(
    csm_capacity["client_count"] > 0,
    csm_capacity["target_rev"] / csm_capacity["client_count"],
    0
)

# Capacity headroom flags
# High concentration = risky (>70% from top 3)
# High required growth = stretch (>20% QoQ)
csm_capacity["concentration_flag"] = np.where(
    csm_capacity["top3_concentration"] > 0.70, "High Risk",
    np.where(csm_capacity["top3_concentration"] > 0.50, "Moderate", "Healthy")
)
csm_capacity["growth_difficulty"] = np.where(
    csm_capacity["required_growth"] > 0.20, "Stretch",
    np.where(csm_capacity["required_growth"] > 0.10, "Achievable", "Conservative")
)

# Composite capacity score: 0-100 (higher = more achievable)
# Penalize high concentration and high required growth
conc_score = np.clip(1.0 - csm_capacity["top3_concentration"], 0, 1) * 50
growth_score = np.clip(1.0 - np.abs(csm_capacity["required_growth"]), 0, 1) * 50
csm_capacity["capacity_score"] = (conc_score + growth_score).round(0).astype(int)

print(f"Clients     : {total_clients}")
print(f"Safe / Risk : {safe_count} / {risk_count}")
print(f"CSM targets sum = {format_usd(alloc_csm['target_rev'].sum())}  (should match {format_usd(total_target_rev)})")
print(f"\nCSM Capacity Summary:")
print(f"  Avg capacity score  : {csm_capacity['capacity_score'].mean():.0f}/100")
print(f"  Stretch targets     : {(csm_capacity['growth_difficulty'] == 'Stretch').sum()} CSMs")
print(f"  High concentration  : {(csm_capacity['concentration_flag'] == 'High Risk').sum()} CSMs")
print("Allocations computed — Q-1 distribution + capacity metrics applied")


In [0]:
# ═══════════════════════════════════════════════════════════════════════
# What-If Simulation — Dual Override: Take Rate + Growth Rate
# ═══════════════════════════════════════════════════════════════════════

_tr_override_raw = dbutils.widgets.get("take_rate_override_pct").strip()
USE_TR_OVERRIDE = len(_tr_override_raw) > 0
OVERRIDE_TR  = float(_tr_override_raw) / 100.0 if USE_TR_OVERRIDE else None

# Growth rate override already applied in Cell 7 to total_target_rev/gp.
# For simulation, we also compute a "what-if" scenario showing delta from baseline.

# Baseline (no overrides) — reconstruct from weighted growth
_baseline_rev = total_base_rev * (1 + weighted_rev_g) * seasonal_index
_baseline_gp  = total_base_gp  * (1 + weighted_gp_g) * seasonal_index
_baseline_tr  = _baseline_gp / _baseline_rev if _baseline_rev else 0

sim = targets.copy()

# ── Apply Take Rate Override ─────────────────────────────────────────
if USE_TR_OVERRIDE:
    sim["sim_tr"] = OVERRIDE_TR
    sim["sim_gp"] = sim["target_revenue"] * OVERRIDE_TR
else:
    sim["sim_tr"] = sim["target_take_rate"]
    sim["sim_gp"] = sim["target_gp"]

sim["sim_guardrail"] = np.where(sim["sim_tr"] >= sim["hist_avg_tr"], "Safe", "Risk")

# ── Compute delta vs baseline (before any overrides) ─────────────────
sim["baseline_rev"] = sim["target_revenue"] * (_baseline_rev / total_target_rev) if total_target_rev > 0 else 0
sim["rev_delta_vs_baseline"] = sim["target_revenue"] - sim["baseline_rev"]

def sim_summary(df, group_cols=None):
    if group_cols:
        g = df.groupby(group_cols).agg(
            target_rev=("target_revenue", "sum"),
            sim_gp=("sim_gp", "sum"),
            baseline_rev=("baseline_rev", "sum"),
            clients=("business_id", "count"),
            safe=("sim_guardrail", lambda x: (x == "Safe").sum()),
            risk=("sim_guardrail", lambda x: (x == "Risk").sum()),
        ).reset_index()
    else:
        g = pd.DataFrame([{
            "target_rev": df["target_revenue"].sum(),
            "sim_gp": df["sim_gp"].sum(),
            "baseline_rev": df["baseline_rev"].sum(),
            "clients": len(df),
            "safe": (df["sim_guardrail"] == "Safe").sum(),
            "risk": (df["sim_guardrail"] == "Risk").sum(),
        }])
    g["sim_tr"] = np.where(g["target_rev"] != 0, g["sim_gp"] / g["target_rev"], 0)
    g["rev_delta"] = g["target_rev"] - g["baseline_rev"]
    g["rev_delta_pct"] = np.where(g["baseline_rev"] != 0, g["rev_delta"] / g["baseline_rev"] * 100, 0)
    return g

sim_overall   = sim_summary(sim)
sim_bg        = sim_summary(sim, ["business_group"])
sim_bu        = sim_summary(sim, ["business_unit"])
sim_motion    = sim_summary(sim, ["account_motion"])
sim_csm       = sim_summary(sim, ["account_manager"])
sim_bu_mot    = sim_summary(sim, ["business_unit", "account_motion"])
sim_csm_mot   = sim_summary(sim, ["account_manager", "account_motion"])

# Summary
tr_label = f"{OVERRIDE_TR*100:.2f}%" if USE_TR_OVERRIDE else "None"
gr_label = f"{float(_gr_override_raw):.2f}%" if GROWTH_OVERRIDE else "None"
print(f"Simulation Overrides:")
print(f"  Take Rate  : {tr_label}")
print(f"  Growth Rate: {gr_label}")
if GROWTH_OVERRIDE or USE_TR_OVERRIDE:
    print(f"  Baseline Rev (no overrides) : {format_usd(_baseline_rev)}")
    print(f"  Simulated Rev               : {format_usd(total_target_rev)}")
    print(f"  Delta                       : {format_usd(total_target_rev - _baseline_rev)}")


In [0]:
# ═══════════════════════════════════════════════════════════════════════
# Global Dashboard Theme — CSS Variables, Animations, Reset
# ═══════════════════════════════════════════════════════════════════════

displayHTML("""
<style>
  :root {
    --bg-primary: #ffffff;
    --bg-secondary: #FAFAF8;
    --bg-hover: #F5F5F2;
    --border-light: #EDEDEA;
    --border-medium: #DDDDD8;
    --text-primary: #1A1A18;
    --text-secondary: #5F5E5A;
    --text-muted: #92918C;
    --text-label: #A09F98;
    --accent-blue: #3B82F6;
    --accent-blue-light: #EFF6FF;
    --accent-green: #10B981;
    --accent-green-light: #ECFDF5;
    --accent-red: #EF4444;
    --accent-red-light: #FEF2F2;
    --accent-amber: #F59E0B;
    --accent-amber-light: #FFFBEB;
    --accent-purple: #8B5CF6;
    --accent-purple-light: #F5F3FF;
    --shadow-sm: 0 1px 2px rgba(0,0,0,0.04);
    --shadow-md: 0 4px 12px rgba(0,0,0,0.04);
    --shadow-lg: 0 8px 24px rgba(0,0,0,0.06);
    --radius-sm: 8px;
    --radius-md: 14px;
    --radius-lg: 20px;
    --font-sans: 'Inter', 'Segoe UI', system-ui, -apple-system, sans-serif;
    --font-mono: 'JetBrains Mono', 'Fira Code', monospace;
  }

  @keyframes fadeInUp {
    from { opacity: 0; transform: translateY(12px); }
    to { opacity: 1; transform: translateY(0); }
  }
  @keyframes slideInRight {
    from { opacity: 0; transform: translateX(-8px); }
    to { opacity: 1; transform: translateX(0); }
  }
  @keyframes barGrow {
    from { width: 0%; }
  }
  @keyframes countUp {
    from { opacity: 0; }
    to { opacity: 1; }
  }
  @keyframes pulseGlow {
    0%, 100% { box-shadow: 0 0 0 0 rgba(59,130,246,0.1); }
    50% { box-shadow: 0 0 0 8px rgba(59,130,246,0); }
  }

  .csm-dash * { box-sizing: border-box; margin: 0; padding: 0; }
  .csm-dash { font-family: var(--font-sans); color: var(--text-primary); max-width: 1440px; width: 100%; }
</style>
""")


In [0]:
# ═══════════════════════════════════════════════════════════════════════
# DASHBOARD — Hero Banner
# ═══════════════════════════════════════════════════════════════════════

_override_tags = []
if GROWTH_OVERRIDE:
    _override_tags.append(f'<span class="hero-tag override">Growth Override: {avg_qoq_growth*100:+.1f}%</span>')
if len(dbutils.widgets.get("take_rate_override_pct").strip()) > 0:
    _override_tags.append(f'<span class="hero-tag override">TR Override: {dbutils.widgets.get("take_rate_override_pct").strip()}%</span>')
if not _override_tags:
    _override_tags.append('<span class="hero-tag">No Overrides Active</span>')
_override_html = " ".join(_override_tags)

displayHTML(f"""
<style>
  .hero-wrap {{
    width: 100%;
    max-width: 1440px;
    box-sizing: border-box;
    background: linear-gradient(135deg, #0f172a 0%, #1e293b 50%, #334155 100%);
    border-radius: var(--radius-lg);
    padding: 36px 44px;
    position: relative;
    overflow: hidden;
    animation: fadeInUp 0.6s ease-out;
  }}
  .hero-wrap::before {{
    content: '';
    position: absolute;
    top: 0; left: 0;
    width: 100%; height: 3px;
    background: linear-gradient(90deg, #3B82F6, #8B5CF6, #EC4899, #F59E0B);
  }}
  .hero-wrap::after {{
    content: '';
    position: absolute;
    top: -50%; right: -10%;
    width: 400px; height: 400px;
    border-radius: 50%;
    background: radial-gradient(circle, rgba(59,130,246,0.08) 0%, transparent 70%);
  }}
  .hero-top {{ display: flex; justify-content: space-between; align-items: flex-start; position: relative; z-index: 1; }}
  .hero-title {{ font-size: 26px; font-weight: 800; color: #fff; letter-spacing: 0.08em; text-transform: uppercase; line-height: 1.2; }}
  .hero-quarter {{ font-size: 40px; font-weight: 900; color: #3B82F6; letter-spacing: -0.02em; line-height: 1; }}
  .hero-subtitle {{ font-size: 13px; color: rgba(255,255,255,0.5); font-weight: 500; margin-top: 8px; }}
  .hero-tags {{ display: flex; gap: 8px; margin-top: 16px; position: relative; z-index: 1; flex-wrap: wrap; }}
  .hero-tag {{
    display: inline-flex; align-items: center; gap: 4px;
    padding: 5px 14px; border-radius: 20px;
    font-size: 11px; font-weight: 600; letter-spacing: 0.02em;
    background: rgba(255,255,255,0.08); color: rgba(255,255,255,0.7);
    border: 1px solid rgba(255,255,255,0.1);
    backdrop-filter: blur(4px);
  }}
  .hero-tag.override {{
    background: rgba(239,68,68,0.15); color: #FCA5A5;
    border-color: rgba(239,68,68,0.3);
  }}
</style>
<div class="csm-dash">
  <div class="hero-wrap">
    <div class="hero-top">
      <div>
        <div class="hero-title">CSM Target Setting</div>
        <div class="hero-subtitle">Automated Quarterly Target Engine &bull; Portfolio-Level Growth &bull; Q-1 Distribution</div>
      </div>
      <div class="hero-quarter">{TARGET_QUARTER}</div>
    </div>
    <div class="hero-tags">
      <span class="hero-tag">L4Q: {', '.join(l4q_quarters)}</span>
      <span class="hero-tag">{total_clients} Clients</span>
      <span class="hero-tag">Base: {last_q}</span>
      {_override_html}
    </div>
  </div>
</div>
""")


In [0]:
# ═══════════════════════════════════════════════════════════════════════
# Executive KPI Cards — Redesigned
# ═══════════════════════════════════════════════════════════════════════

growth_pct = ((total_target_rev / total_base_rev) - 1) * 100 if total_base_rev else 0
target_tr = total_target_gp / total_target_rev if total_target_rev else 0
_method_label = "Override" if GROWTH_OVERRIDE else "Weighted QoQ"
_capacity_avg = csm_capacity["capacity_score"].mean()

kpis = [
    {
        "label": "Base Revenue",
        "value": format_usd(total_base_rev),
        "sub": f"Q-1: {last_q}",
        "accent": "#64748B",
        "icon": "&#9679;"
    },
    {
        "label": "Target Revenue",
        "value": format_usd(total_target_rev),
        "sub": f"{_method_label}: {avg_qoq_growth*100:+.1f}%",
        "accent": "#3B82F6",
        "icon": "&#9650;"
    },
    {
        "label": "Target Gross Profit",
        "value": format_usd(total_target_gp),
        "sub": f"Take Rate: {fmt_pct(target_tr)}",
        "accent": "#10B981",
        "icon": "&#9733;"
    },
    {
        "label": "Growth",
        "value": f"{growth_pct:+.1f}%",
        "sub": f"Seasonal: {'x' + f'{seasonal_index:.2f}' if HAS_SEASONAL else 'N/A'}",
        "accent": "#8B5CF6" if growth_pct >= 0 else "#EF4444",
        "icon": "&#8593;" if growth_pct >= 0 else "&#8595;"
    },
    {
        "label": "Momentum",
        "value": momentum_label,
        "sub": f"{momentum*100:+.2f}pp shift",
        "accent": "#10B981" if momentum > 0 else ("#EF4444" if momentum < 0 else "#64748B"),
        "icon": "&#8599;" if momentum > 0 else ("&#8600;" if momentum < 0 else "&#8594;")
    },
    {
        "label": "Guardrails",
        "value": f"{safe_count}S / {risk_count}R",
        "sub": f"Capacity avg: {_capacity_avg:.0f}/100",
        "accent": "#10B981" if safe_count > risk_count else "#EF4444",
        "icon": "&#9632;"
    },
]
_kpi_payload = json.dumps(kpis)

displayHTML(f"""
<style>
  .kpi-grid {{
    display: grid;
    grid-template-columns: repeat(6, 1fr);
    gap: 12px;
    margin: 16px 0;
    animation: fadeInUp 0.5s ease-out 0.1s both;
  }}
  .kpi-card {{
    background: var(--bg-primary);
    border: 1px solid var(--border-light);
    border-radius: var(--radius-md);
    padding: 20px 16px;
    position: relative;
    overflow: hidden;
    transition: all 0.25s ease;
  }}
  .kpi-card:hover {{
    border-color: var(--border-medium);
    box-shadow: var(--shadow-md);
    transform: translateY(-2px);
  }}
  .kpi-card::before {{
    content: '';
    position: absolute;
    top: 0; left: 0;
    width: 100%; height: 3px;
    border-radius: var(--radius-md) var(--radius-md) 0 0;
  }}
  .kpi-icon {{
    font-size: 14px;
    width: 28px; height: 28px;
    border-radius: 8px;
    display: flex; align-items: center; justify-content: center;
    margin-bottom: 12px;
  }}
  .kpi-label {{
    font-size: 10px;
    font-weight: 700;
    color: var(--text-label);
    text-transform: uppercase;
    letter-spacing: 0.1em;
    margin-bottom: 6px;
  }}
  .kpi-value {{
    font-size: 22px;
    font-weight: 800;
    color: var(--text-primary);
    letter-spacing: -0.02em;
    line-height: 1.1;
  }}
  .kpi-sub {{
    font-size: 11px;
    font-weight: 500;
    color: var(--text-muted);
    margin-top: 6px;
  }}
</style>
<div class="csm-dash">
  <div class="kpi-grid" id="kpi-grid-exec"></div>
</div>
<script>
  (function() {{
    const data = {_kpi_payload};
    const ct = document.getElementById('kpi-grid-exec');
    ct.innerHTML = data.map(k => `
      <div class="kpi-card" style="--kpi-accent:${{k.accent}}">
        <div style="position:absolute;top:0;left:0;width:100%;height:3px;background:${{k.accent}};border-radius:14px 14px 0 0;"></div>
        <div class="kpi-icon" style="background:${{k.accent}}15;color:${{k.accent}}">${{k.icon}}</div>
        <div class="kpi-label">${{k.label}}</div>
        <div class="kpi-value">${{k.value}}</div>
        <div class="kpi-sub">${{k.sub}}</div>
      </div>
    `).join('');
  }})();
</script>
""")


In [0]:
# ═══════════════════════════════════════════════════════════════════════
# Growth Intelligence Panel — Visual methodology breakdown
# ═══════════════════════════════════════════════════════════════════════

# Prepare data for the growth pipeline visualization
_growth_steps = [
    {
        "step": "1",
        "title": "QoQ Growth Rates",
        "desc": f"L4Q portfolio QoQ: {', '.join([f'{g*100:+.1f}%' for g in port_q['rev_g'].dropna().values])}",
        "value": f"{simple_rev_g*100:+.2f}%",
        "label": "Simple Avg",
    },
    {
        "step": "2",
        "title": "Recency Weighting",
        "desc": "Exponential weights — recent quarters count more (2^n)",
        "value": f"{weighted_rev_g*100:+.2f}%",
        "label": "Weighted QoQ",
    },
    {
        "step": "3",
        "title": "Momentum Detection",
        "desc": f"Recent vs older half: {momentum*100:+.2f}pp → {momentum_label}",
        "value": momentum_label,
        "label": "Trend",
    },
    {
        "step": "4",
        "title": f"Seasonal Index (Q{int(TARGET_QUARTER[-1])})",
        "desc": f"Avg same-Q rev ÷ avg all-Q rev = {seasonal_index:.3f}" if HAS_SEASONAL else "Insufficient YoY data",
        "value": f"x{seasonal_index:.2f}" if HAS_SEASONAL else "x1.00",
        "label": "Multiplier",
    },
    {
        "step": "5",
        "title": "Final Target",
        "desc": f"Base ({last_q}) × (1 + growth) × seasonal",
        "value": format_usd(total_target_rev),
        "label": "Target Revenue",
    },
]

_steps_payload = json.dumps(_growth_steps)

# Historical take rate info
_tr_hist = fmt_pct(HIST_AVG_TR)
_tr_target = fmt_pct(target_tr)
_tr_status = "above" if target_tr >= HIST_AVG_TR else "below"
_tr_color = "#10B981" if target_tr >= HIST_AVG_TR else "#EF4444"

displayHTML(f"""
<style>
  .growth-panel {{
    background: var(--bg-primary);
    border: 1px solid var(--border-light);
    border-radius: var(--radius-lg);
    padding: 28px 32px;
    margin: 16px 0;
    animation: fadeInUp 0.5s ease-out 0.2s both;
  }}
  .growth-panel-header {{
    display: flex;
    justify-content: space-between;
    align-items: center;
    margin-bottom: 24px;
    padding-bottom: 16px;
    border-bottom: 1px solid var(--border-light);
  }}
  .gp-title {{
    font-size: 17px;
    font-weight: 800;
    color: var(--text-primary);
  }}
  .gp-badge {{
    font-size: 11px;
    font-weight: 700;
    padding: 4px 12px;
    border-radius: 20px;
    background: var(--accent-blue-light);
    color: var(--accent-blue);
  }}
  .growth-pipeline {{
    display: flex;
    gap: 0;
    align-items: stretch;
  }}
  .gp-step {{
    flex: 1;
    padding: 16px 14px;
    position: relative;
    border-right: 1px solid var(--border-light);
    transition: background 0.2s;
  }}
  .gp-step:last-child {{ border-right: none; }}
  .gp-step:hover {{ background: var(--bg-secondary); }}
  .gp-step-num {{
    width: 24px; height: 24px;
    border-radius: 50%;
    background: var(--accent-blue);
    color: #fff;
    font-size: 11px;
    font-weight: 800;
    display: flex; align-items: center; justify-content: center;
    margin-bottom: 10px;
  }}
  .gp-step:last-child .gp-step-num {{ background: #10B981; }}
  .gp-step-title {{ font-size: 12px; font-weight: 700; color: var(--text-primary); margin-bottom: 4px; }}
  .gp-step-desc {{ font-size: 10px; color: var(--text-muted); line-height: 1.4; margin-bottom: 10px; min-height: 28px; }}
  .gp-step-value {{ font-size: 18px; font-weight: 800; color: var(--accent-blue); }}
  .gp-step:last-child .gp-step-value {{ color: #10B981; }}
  .gp-step-label {{ font-size: 9px; font-weight: 700; color: var(--text-label); text-transform: uppercase; letter-spacing: 0.08em; margin-top: 2px; }}
  .gp-arrow {{
    display: flex;
    align-items: center;
    color: var(--border-medium);
    font-size: 16px;
    padding: 0 2px;
  }}
  .tr-strip {{
    display: flex;
    gap: 16px;
    margin-top: 20px;
    padding-top: 16px;
    border-top: 1px solid var(--border-light);
  }}
  .tr-item {{
    display: flex;
    align-items: center;
    gap: 8px;
    font-size: 12px;
    font-weight: 600;
    color: var(--text-secondary);
  }}
  .tr-dot {{
    width: 8px; height: 8px;
    border-radius: 50%;
  }}
</style>
<div class="csm-dash">
  <div class="growth-panel">
    <div class="growth-panel-header">
      <span class="gp-title">Growth Methodology Pipeline</span>
      <span class="gp-badge">{'OVERRIDE ACTIVE' if GROWTH_OVERRIDE else 'Auto-Computed'}</span>
    </div>
    <div class="growth-pipeline" id="growth-pipeline"></div>
    <div class="tr-strip">
      <div class="tr-item"><div class="tr-dot" style="background:#64748B;"></div>Historical L4Q Take Rate: {_tr_hist}</div>
      <div class="tr-item"><div class="tr-dot" style="background:{_tr_color};"></div>Target Take Rate: {_tr_target} ({_tr_status} historical)</div>
      <div class="tr-item"><div class="tr-dot" style="background:#8B5CF6;"></div>Method: Recency-Weighted Exponential</div>
    </div>
  </div>
</div>
<script>
  (function() {{
    const steps = {_steps_payload};
    const ct = document.getElementById('growth-pipeline');
    ct.innerHTML = steps.map((s, i) => `
      <div class="gp-step" style="animation:slideInRight 0.3s ease ${{i * 0.1}}s both">
        <div class="gp-step-num">${{s.step}}</div>
        <div class="gp-step-title">${{s.title}}</div>
        <div class="gp-step-desc">${{s.desc}}</div>
        <div class="gp-step-value">${{s.value}}</div>
        <div class="gp-step-label">${{s.label}}</div>
      </div>
    `).join('');
  }})();
</script>
""")


In [0]:
# ═══════════════════════════════════════════════════════════════════════
# QoQ Revenue & GP Trend (native chart)
# ═══════════════════════════════════════════════════════════════════════

target_row = pd.DataFrame([{
    "quarter": TARGET_QUARTER,
    "gross_revenue": total_target_rev,
    "gross_profit": total_target_gp,
    "is_target": True,
}])
trend = q_overall.copy()
trend["is_target"] = False
trend = pd.concat([trend, target_row], ignore_index=True).sort_values("quarter")
trend["type"] = trend["is_target"].map({True: "Target", False: "Actual"})
trend["take_rate_pct"] = trend.apply(
    lambda r: round(r["gross_profit"] / r["gross_revenue"] * 100, 2) if r["gross_revenue"] else 0, axis=1
)
trend_melted = trend.melt(
    id_vars=["quarter", "type"],
    value_vars=["gross_revenue", "gross_profit"],
    var_name="metric", value_name="amount"
)
trend_melted["metric"] = trend_melted["metric"].map({"gross_revenue": "Revenue", "gross_profit": "Gross Profit"})
display(trend_melted)


In [0]:
# ═══════════════════════════════════════════════════════════════════════
# Compute Monthly Aggregations + QoQ & MoM Actuals & Targets Data
# ═══════════════════════════════════════════════════════════════════════

# Monthly aggregations at all granularities (for MoM view)
rev["month_str"] = rev["txn_date"].dt.strftime("%Y-%m")

# Helper to compute period-over-period for a granularity
def compute_pop(df, group_cols, period_col, value_cols=["gross_revenue", "gross_profit"]):
    """Compute period-over-period growth for given granularity."""
    agg_dict = {c: (c, "sum") for c in value_cols}
    agg_dict["count"] = ("business_id", "nunique")
    grouped = df.groupby(group_cols + [period_col]).agg(**agg_dict).reset_index()
    grouped = grouped.sort_values(group_cols + [period_col])
    for vc in value_cols:
        grouped[f"{vc}_prev"] = grouped.groupby(group_cols)[vc].shift(1) if group_cols else grouped[vc].shift(1)
        grouped[f"{vc}_chg"] = grouped[vc] - grouped[f"{vc}_prev"]
        grouped[f"{vc}_pct"] = np.where(
            grouped[f"{vc}_prev"].abs() > 0,
            (grouped[vc] - grouped[f"{vc}_prev"]) / grouped[f"{vc}_prev"].abs() * 100,
            0
        )
    grouped["take_rate"] = np.where(grouped["gross_revenue"] != 0, grouped["gross_profit"] / grouped["gross_revenue"] * 100, 0)
    return grouped

# ── QoQ data at all granularities ───────────────────────────────────────
qoq_overall = compute_pop(rev, [], "quarter")
qoq_bg      = compute_pop(rev, ["business_group"], "quarter")
qoq_bu      = compute_pop(rev, ["business_unit"], "quarter")
qoq_csm     = compute_pop(rev, ["account_manager"], "quarter")
qoq_motion  = compute_pop(rev, ["account_motion"], "quarter")
qoq_bu_mot  = compute_pop(rev, ["business_unit", "account_motion"], "quarter")
qoq_csm_mot = compute_pop(rev, ["account_manager", "account_motion"], "quarter")

# ── MoM data at all granularities ───────────────────────────────────────
mom_overall = compute_pop(rev, [], "month_str")
mom_bg      = compute_pop(rev, ["business_group"], "month_str")
mom_bu      = compute_pop(rev, ["business_unit"], "month_str")
mom_csm     = compute_pop(rev, ["account_manager"], "month_str")
mom_motion  = compute_pop(rev, ["account_motion"], "month_str")
mom_bu_mot  = compute_pop(rev, ["business_unit", "account_motion"], "month_str")
mom_csm_mot = compute_pop(rev, ["account_manager", "account_motion"], "month_str")

# ── Build target row for QoQ (append to each granularity) ──────────
# Target quarter targets at each granularity for comparison
def append_targets_qoq(pop_df, alloc_df, group_cols, period_col="quarter"):
    """Append target quarter row to QoQ data, with calculated QoQ% and take rate."""
    result = pop_df.assign(is_target=False)
    if len(group_cols) == 0:
        # Overall
        prev_row = result[result[period_col] == last_q]
        prev_rev = prev_row["gross_revenue"].values[0] if len(prev_row) > 0 else 0
        prev_gp  = prev_row["gross_profit"].values[0] if len(prev_row) > 0 else 0
        tgt_row = {
            period_col: TARGET_QUARTER,
            "gross_revenue": total_target_rev,
            "gross_profit": total_target_gp,
            "count": total_clients,
            "is_target": True,
            "take_rate": total_target_gp / total_target_rev * 100 if total_target_rev else 0,
            "gross_revenue_prev": prev_rev,
            "gross_profit_prev": prev_gp,
            "gross_revenue_chg": total_target_rev - prev_rev,
            "gross_profit_chg": total_target_gp - prev_gp,
            "gross_revenue_pct": ((total_target_rev - prev_rev) / prev_rev * 100) if prev_rev else 0,
            "gross_profit_pct": ((total_target_gp - prev_gp) / prev_gp * 100) if prev_gp else 0,
        }
        return pd.concat([result, pd.DataFrame([tgt_row])], ignore_index=True)

    tgt_rows = []
    for _, r in alloc_df.iterrows():
        # Find previous actuals for this group
        prev = result
        for gc in group_cols:
            prev = prev[prev[gc] == r[gc]]
        prev = prev[prev[period_col] == last_q]
        prev_rev = prev["gross_revenue"].values[0] if len(prev) > 0 else 0
        prev_gp  = prev["gross_profit"].values[0] if len(prev) > 0 else 0
        tr = r.get("l4q_tr", target_tr)
        tgt_row = {
            period_col: TARGET_QUARTER,
            "is_target": True,
            "count": 0,
            "gross_revenue": r["target_rev"],
            "gross_profit": r["target_rev"] * tr,
            "take_rate": tr * 100,
            "gross_revenue_prev": prev_rev,
            "gross_profit_prev": prev_gp,
            "gross_revenue_chg": r["target_rev"] - prev_rev,
            "gross_profit_chg": r["target_rev"] * tr - prev_gp,
            "gross_revenue_pct": ((r["target_rev"] - prev_rev) / prev_rev * 100) if prev_rev else 0,
            "gross_profit_pct": ((r["target_rev"] * tr - prev_gp) / prev_gp * 100) if prev_gp else 0,
        }
        for gc in group_cols:
            tgt_row[gc] = r[gc]
        tgt_rows.append(tgt_row)

    return pd.concat([result, pd.DataFrame(tgt_rows)], ignore_index=True)

qoq_overall_t = append_targets_qoq(qoq_overall, None, [])
qoq_bg_t      = append_targets_qoq(qoq_bg, alloc_bg, ["business_group"])
qoq_bu_t      = append_targets_qoq(qoq_bu, alloc_bu, ["business_unit"])
qoq_csm_t     = append_targets_qoq(qoq_csm, alloc_csm, ["account_manager"])
qoq_motion_t  = append_targets_qoq(qoq_motion, alloc_motion, ["account_motion"])
qoq_bu_mot_t  = append_targets_qoq(qoq_bu_mot, alloc_bu_mot, ["business_unit", "account_motion"])
qoq_csm_mot_t = append_targets_qoq(qoq_csm_mot, alloc_csm_mot, ["account_manager", "account_motion"])

# ── MoM targets: use last available month as base for target row ──────
def append_targets_mom(pop_df, alloc_df, group_cols, period_col="month_str"):
    """Append target month row to MoM data, with calculated MoM% and take rate."""
    result = pop_df.assign(is_target=False)
    # Find last available month
    last_month = result[period_col].max()
    if len(group_cols) == 0:
        prev_row = result[result[period_col] == last_month]
        prev_rev = prev_row["gross_revenue"].values[0] if len(prev_row) > 0 else 0
        prev_gp  = prev_row["gross_profit"].values[0] if len(prev_row) > 0 else 0
        tgt_row = {
            period_col: TARGET_QUARTER,
            "gross_revenue": total_target_rev,
            "gross_profit": total_target_gp,
            "count": total_clients,
            "is_target": True,
            "take_rate": total_target_gp / total_target_rev * 100 if total_target_rev else 0,
            "gross_revenue_prev": prev_rev,
            "gross_profit_prev": prev_gp,
            "gross_revenue_chg": total_target_rev - prev_rev,
            "gross_profit_chg": total_target_gp - prev_gp,
            "gross_revenue_pct": ((total_target_rev - prev_rev) / prev_rev * 100) if prev_rev else 0,
            "gross_profit_pct": ((total_target_gp - prev_gp) / prev_gp * 100) if prev_gp else 0,
        }
        return pd.concat([result, pd.DataFrame([tgt_row])], ignore_index=True)

    tgt_rows = []
    for _, r in alloc_df.iterrows():
        prev = result
        for gc in group_cols:
            prev = prev[prev[gc] == r[gc]]
        prev = prev[prev[period_col] == last_month]
        prev_rev = prev["gross_revenue"].values[0] if len(prev) > 0 else 0
        prev_gp  = prev["gross_profit"].values[0] if len(prev) > 0 else 0
        tr = r.get("l4q_tr", target_tr)
        tgt_row = {
            period_col: TARGET_QUARTER,
            "is_target": True,
            "count": 0,
            "gross_revenue": r["target_rev"],
            "gross_profit": r["target_rev"] * tr,
            "take_rate": tr * 100,
            "gross_revenue_prev": prev_rev,
            "gross_profit_prev": prev_gp,
            "gross_revenue_chg": r["target_rev"] - prev_rev,
            "gross_profit_chg": r["target_rev"] * tr - prev_gp,
            "gross_revenue_pct": ((r["target_rev"] - prev_rev) / prev_rev * 100) if prev_rev else 0,
            "gross_profit_pct": ((r["target_rev"] * tr - prev_gp) / prev_gp * 100) if prev_gp else 0,
        }
        for gc in group_cols:
            tgt_row[gc] = r[gc]
        tgt_rows.append(tgt_row)

    return pd.concat([result, pd.DataFrame(tgt_rows)], ignore_index=True)

mom_overall_t   = append_targets_mom(mom_overall, None, [])
mom_bg_t        = append_targets_mom(mom_bg, alloc_bg, ["business_group"])
mom_bu_t        = append_targets_mom(mom_bu, alloc_bu, ["business_unit"])
mom_csm_t       = append_targets_mom(mom_csm, alloc_csm, ["account_manager"])
mom_motion_t    = append_targets_mom(mom_motion, alloc_motion, ["account_motion"])
mom_bu_mot_t    = append_targets_mom(mom_bu_mot, alloc_bu_mot, ["business_unit", "account_motion"])
mom_csm_mot_t   = append_targets_mom(mom_csm_mot, alloc_csm_mot, ["account_manager", "account_motion"])

# ── Serialize for JS ─────────────────────────────────────────────────
def pop_to_json(df, group_cols, period_col):
    """Convert pop data to JSON-serializable list."""
    rows = []
    for _, r in df.sort_values(group_cols + [period_col]).iterrows():
        d = {
            "period": str(r[period_col]),
            "rev": float(r.get("gross_revenue", 0) or 0),
            "gp": float(r.get("gross_profit", 0) or 0),
            "tr": round(float(r.get("take_rate", 0) or 0), 2) if "take_rate" in r.index else 0,
            "rev_pct": round(float(r.get("gross_revenue_pct", 0) or 0), 2) if "gross_revenue_pct" in r.index else None,
            "gp_pct": round(float(r.get("gross_profit_pct", 0) or 0), 2) if "gross_profit_pct" in r.index else None,
            "is_target": bool(r.get("is_target", False)),
        }
        for gc in group_cols:
            d[gc] = str(r[gc])
        rows.append(d)
    return rows

_pop_data = {
    "qoq": {
        "Overall": pop_to_json(qoq_overall_t, [], "quarter"),
        "Business Group": pop_to_json(qoq_bg_t, ["business_group"], "quarter"),
        "Business Unit": pop_to_json(qoq_bu_t, ["business_unit"], "quarter"),
        "CSM": pop_to_json(qoq_csm_t, ["account_manager"], "quarter"),
        "Motion": pop_to_json(qoq_motion_t, ["account_motion"], "quarter"),
        "BU x Motion": pop_to_json(qoq_bu_mot_t, ["business_unit", "account_motion"], "quarter"),
        "CSM x Motion": pop_to_json(qoq_csm_mot_t, ["account_manager", "account_motion"], "quarter"),
    },
    "mom": {
        "Overall": pop_to_json(mom_overall_t, [], "month_str"),
        "Business Group": pop_to_json(mom_bg_t, ["business_group"], "month_str"),
        "Business Unit": pop_to_json(mom_bu_t, ["business_unit"], "month_str"),
        "CSM": pop_to_json(mom_csm_t, ["account_manager"], "month_str"),
        "Motion": pop_to_json(mom_motion_t, ["account_motion"], "month_str"),
        "BU x Motion": pop_to_json(mom_bu_mot_t, ["business_unit", "account_motion"], "month_str"),
        "CSM x Motion": pop_to_json(mom_csm_mot_t, ["account_manager", "account_motion"], "month_str"),
    },
}

_pop_payload = json.dumps(_pop_data, default=str)
print(f"QoQ/MoM data prepared. QoQ periods: {sorted(qoq_overall_t['quarter'].unique())}")
print(f"MoM periods: {sorted(mom_overall_t['month_str'].unique())}")

In [0]:
displayHTML(f"""
<style>
  .sec-banner {{
    display: flex;
    align-items: center;
    gap: 16px;
    padding: 20px 0;
    margin: 24px 0 8px;
    border-bottom: 2px solid #1A1A18;
    animation: fadeInUp 0.4s ease-out;
  }}
  .sec-dot {{
    width: 10px; height: 10px;
    border-radius: 50%;
    background: #3B82F6;
  }}
  .sec-title {{
    font-size: 20px;
    font-weight: 800;
    color: #1A1A18;
    letter-spacing: -0.01em;
    font-family: var(--font-sans);
  }}
  .sec-subtitle {{
    font-size: 12px;
    font-weight: 600;
    color: #92918C;
    margin-left: auto;
  }}
</style>
<div class="csm-dash">
  <div class="sec-banner">
    <div class="sec-dot"></div>
    <div class="sec-title">Actuals & Targets — QoQ / MoM View</div>
    <div class="sec-subtitle">Toggle between Quarter-over-Quarter and Month-over-Month</div>
  </div>
</div>
""")

In [0]:
# ═══════════════════════════════════════════════════════════════════════
# QoQ & MoM Toggle View — All Granularities
# ═══════════════════════════════════════════════════════════════════════

displayHTML(f"""
<style>
  .pop-wrap {{
    font-family: var(--font-sans);
    animation: fadeInUp 0.4s ease-out;
  }}
  /* Toggle Bar */
  .pop-toggle-bar {{
    display: flex; align-items: center; gap: 4px;
    background: var(--bg-secondary); border: 1px solid var(--border-light);
    border-radius: 12px; padding: 4px; width: fit-content;
    margin-bottom: 20px;
  }}
  .pop-toggle-btn {{
    padding: 8px 24px; border-radius: 10px; border: none;
    font-size: 13px; font-weight: 700; cursor: pointer;
    transition: all 0.25s; background: transparent; color: var(--text-muted);
    font-family: var(--font-sans);
  }}
  .pop-toggle-btn.active {{
    background: var(--bg-primary); color: var(--text-primary);
    box-shadow: var(--shadow-sm); border: 1px solid var(--border-light);
  }}
  .pop-toggle-btn:hover:not(.active) {{ color: var(--text-secondary); }}

  /* Granularity Tabs */
  .pop-tabs {{
    display: flex; gap: 2px; margin-bottom: 16px;
    border-bottom: 1px solid var(--border-light);
    overflow-x: auto;
  }}
  .pop-tab {{
    padding: 10px 18px; font-size: 12px; font-weight: 600;
    color: var(--text-muted); cursor: pointer; border: none;
    background: transparent; border-bottom: 2px solid transparent;
    transition: all 0.2s; white-space: nowrap;
    font-family: var(--font-sans);
  }}
  .pop-tab.active {{
    color: var(--accent-blue); border-bottom-color: var(--accent-blue);
    font-weight: 700;
  }}
  .pop-tab:hover:not(.active) {{ color: var(--text-secondary); background: var(--bg-secondary); }}

  /* Table */
  .pop-table-wrap {{
    background: var(--bg-primary); border: 1px solid var(--border-light);
    border-radius: var(--radius-md);
    box-shadow: var(--shadow-sm);
    overflow: hidden;
  }}
  .pop-tbl-scroll {{ max-height: 560px; overflow: auto; }}
  table.pop-tbl {{ width: 100%; border-collapse: collapse; font-size: 12px; }}
  table.pop-tbl thead {{
  position: sticky;
  top: 0;
  z-index: 10;                         /* raised so it clears tbody rows */
}}

table.pop-tbl th {{
  text-align: left;
  font-size: 10px;
  font-weight: 700;
  color: var(--text-label);
  text-transform: uppercase;
  letter-spacing: 0.06em;
  padding: 12px 14px;
  background: #e8edf5;                 /* solid — swap for any hex you prefer */
  border-bottom: 2px solid var(--border-light);
  /* belt-and-suspenders: some browsers need this on <th> directly */
  position: sticky;
  top: 0;
}}
  table.pop-tbl td {{
    padding: 10px 14px; border-bottom: 1px solid var(--border-light);
    font-variant-numeric: tabular-nums;
  }}
  table.pop-tbl tbody tr:nth-child(even) {{ background: var(--bg-secondary); }}
  table.pop-tbl tbody tr:hover {{ background: var(--bg-hover); }}
  table.pop-tbl .num {{ text-align: right; font-weight: 600; }}
  .pop-chg-pos {{ color: #047857; font-weight: 700; }}
  .pop-chg-neg {{ color: #b91c1c; font-weight: 700; }}
  .pop-target-row {{ background: #eff6ff !important; }}
  .pop-target-row td {{ font-weight: 700; color: var(--accent-blue); }}
  .pop-target-badge {{
    display: inline-block; padding: 2px 8px; border-radius: 4px;
    font-size: 9px; font-weight: 800; background: #dbeafe; color: #1d4ed8;
    text-transform: uppercase; letter-spacing: 0.04em;
  }}
  .pop-empty {{
    padding: 40px; text-align: center; color: var(--text-muted);
    font-size: 13px;
  }}
</style>
<div class="csm-dash">
  <div class="pop-wrap">
    <div class="pop-toggle-bar">
      <button class="pop-toggle-btn active" onclick="switchMode('qoq')">QoQ — Quarter over Quarter</button>
      <button class="pop-toggle-btn" onclick="switchMode('mom')">MoM — Month over Month</button>
    </div>
    <div class="pop-tabs" id="pop-tabs"></div>
    <div class="pop-table-wrap">
      <div class="pop-tbl-scroll">
        <table class="pop-tbl">
          <thead id="pop-thead"></thead>
          <tbody id="pop-tbody"></tbody>
        </table>
      </div>
    </div>
  </div>
</div>
<script>
(function() {{
  const allData = {_pop_payload};
  const grans = ["Overall","Business Group","Business Unit","CSM","Motion","BU x Motion","CSM x Motion"];
  const granKeys = {{"Overall":[],"Business Group":["business_group"],"Business Unit":["business_unit"],
    "CSM":["account_manager"],"Motion":["account_motion"],
    "BU x Motion":["business_unit","account_motion"],"CSM x Motion":["account_manager","account_motion"]}};

  let curMode = "qoq";
  let curGran = "Overall";

  function fmtUsd(v) {{
    if (!v && v !== 0) return "$0";
    const a = Math.abs(v);
    if (a >= 1e9) return "$" + (v/1e9).toFixed(2) + "B";
    if (a >= 1e6) return "$" + (v/1e6).toFixed(2) + "M";
    if (a >= 1e3) return "$" + (v/1e3).toFixed(2) + "K";
    return "$" + v.toFixed(2);
  }}

  function fmtChg(v) {{
    if (v === null || v === undefined) return '<span style="color:var(--text-muted)">—</span>';
    const cls = v > 0 ? "pop-chg-pos" : v < 0 ? "pop-chg-neg" : "";
    const sign = v > 0 ? "+" : "";
    return `<span class="${{cls}}">${{sign}}${{v.toFixed(1)}}%</span>`;
  }}

  function switchMode(mode) {{
    curMode = mode;
    document.querySelectorAll('.pop-toggle-btn').forEach(b => b.classList.remove('active'));
    document.querySelectorAll('.pop-toggle-btn').forEach(b => {{
      if ((mode === 'qoq' && b.textContent.includes('QoQ')) || (mode === 'mom' && b.textContent.includes('MoM')))
        b.classList.add('active');
    }});
    render();
  }}
  window.switchMode = switchMode;

  function switchGran(gran) {{
    curGran = gran;
    document.querySelectorAll('.pop-tab').forEach(t => t.classList.toggle('active', t.dataset.gran === gran));
    render();
  }}
  window.switchGran = switchGran;

  function renderTabs() {{
    const ct = document.getElementById('pop-tabs');
    ct.innerHTML = grans.map(g =>
      `<button class="pop-tab ${{g === curGran ? 'active' : ''}}" data-gran="${{g}}" onclick="switchGran('${{g}}')">${{g}}</button>`
    ).join('');
  }}

  function render() {{
    renderTabs();
    const data = allData[curMode][curGran] || [];
    const keys = granKeys[curGran] || [];
    const thead = document.getElementById('pop-thead');
    const tbody = document.getElementById('pop-tbody');

    if (data.length === 0) {{
      thead.innerHTML = '';
      tbody.innerHTML = '<tr><td class="pop-empty" colspan="10">No data for this view</td></tr>';
      return;
    }}

    // Build header
    let hCols = keys.map(k => `<th>${{k.replace('account_','').replace('business_','').replace('_',' ').toUpperCase()}}</th>`).join('');
    hCols += `<th>Period</th>
              <th class="num" style="text-align:right">Revenue</th>
              <th class="num" style="text-align:right">Gross Profit</th>
              <th class="num" style="text-align:right">Take Rate</th>
              <th class="num" style="text-align:right">Rev ${{curMode === 'qoq' ? 'QoQ' : 'MoM'}} %</th>
              <th class="num" style="text-align:right">GP ${{curMode === 'qoq' ? 'QoQ' : 'MoM'}} %</th>`;
    thead.innerHTML = `<tr>${{hCols}}</tr>`;

    // Group by dimension keys
    let grouped = {{}};
    data.forEach(r => {{
      const gk = keys.map(k => r[k] || '').join('|') || '__all__';
      if (!grouped[gk]) grouped[gk] = [];
      grouped[gk].push(r);
    }});

    let html = '';
    for (const [gk, rows] of Object.entries(grouped)) {{
      rows.sort((a,b) => a.period.localeCompare(b.period));
      rows.forEach((r, ri) => {{
        const isTarget = r.is_target;
        const rowCls = isTarget ? 'pop-target-row' : '';
        let cells = keys.map(k => `<td style="font-weight:600">${{r[k] || ''}}</td>`).join('');
        const periodLabel = isTarget ? `${{r.period}} <span class="pop-target-badge">Target</span>` : r.period;
        cells += `<td>${{periodLabel}}</td>`;
        cells += `<td class="num">${{fmtUsd(r.rev)}}</td>`;
        cells += `<td class="num">${{fmtUsd(r.gp)}}</td>`;
        const trVal = r.tr !== undefined && r.tr !== null ? r.tr.toFixed(2) + '%' : (r.rev > 0 ? ((r.gp/r.rev)*100).toFixed(2) + '%' : '0%');
        cells += `<td class="num">${{trVal}}</td>`;
        cells += `<td class="num">${{fmtChg(r.rev_pct)}}</td>`;
        cells += `<td class="num">${{fmtChg(r.gp_pct)}}</td>`;
        html += `<tr class="${{rowCls}}" style="animation:fadeInUp 0.15s ease ${{ri*0.02}}s both">${{cells}}</tr>`;
      }});
    }}
    tbody.innerHTML = html;
  }}

  renderTabs();
  render();
}})();
</script>
""")


In [0]:
displayHTML(f"""
<style>
  .sec-banner {{
    display: flex;
    align-items: center;
    gap: 16px;
    padding: 20px 0;
    margin: 24px 0 8px;
    border-bottom: 2px solid #1A1A18;
    animation: fadeInUp 0.4s ease-out;
  }}
  .sec-dot {{
    width: 10px; height: 10px;
    border-radius: 50%;
    background: #3B82F6;
  }}
  .sec-title {{
    font-size: 20px;
    font-weight: 800;
    color: #1A1A18;
    letter-spacing: -0.01em;
    font-family: var(--font-sans);
  }}
  .sec-subtitle {{
    font-size: 12px;
    font-weight: 600;
    color: #92918C;
    margin-left: auto;
  }}
</style>
<div class="csm-dash">
  <div class="sec-banner">
    <div class="sec-dot"></div>
    <div class="sec-title">Target Allocations</div>
    <div class="sec-subtitle">{TARGET_QUARTER} — Q-1 Distribution</div>
  </div>
</div>
""")

In [0]:
# ═══════════════════════════════════════════════════════════════════════
# Target Allocation — BG + BU Side by Side
# ═══════════════════════════════════════════════════════════════════════

def _alloc_items(df, label_col, color_map):
    total = df["target_rev"].sum()
    items = []
    for i, row in df.iterrows():
        lbl = str(row[label_col])
        pct = round(row["target_rev"] / total * 100, 1) if total > 0 else 0
        items.append({
            "label": lbl,
            "pct": pct,
            "amt": format_usd(row["target_rev"]),
            "tr": round(row.get("l4q_tr", 0) * 100, 2),
            "color": color_map.get(lbl, PALETTE[i % len(PALETTE)]),
        })
    return items

_bg_items = _alloc_items(alloc_bg, "business_group", BG_COLORS)
_bu_items = _alloc_items(alloc_bu, "business_unit", BU_COLORS)
_bg_payload = json.dumps(_bg_items)
_bu_payload = json.dumps(_bu_items)

displayHTML(f"""
<style>
  .alloc-duo {{
    display: grid;
    grid-template-columns: 1fr 1fr;
    gap: 16px;
    margin: 12px 0;
    animation: fadeInUp 0.4s ease-out;
  }}
  .alloc-panel {{
    background: var(--bg-primary);
    border: 1px solid var(--border-light);
    border-radius: var(--radius-md);
    padding: 24px;
    box-shadow: var(--shadow-sm);
    font-family: var(--font-sans);
    transition: box-shadow 0.3s;
  }}
  .alloc-panel:hover {{ box-shadow: var(--shadow-md); }}
  .ap-header {{ margin-bottom: 16px; padding-bottom: 10px; border-bottom: 1px solid var(--border-light); }}
  .ap-title {{ font-size: 15px; font-weight: 800; color: var(--text-primary); margin: 0 0 2px; }}
  .ap-sub {{ font-size: 10px; font-weight: 600; color: var(--text-label); text-transform: uppercase; letter-spacing: 0.06em; }}
  .ap-row {{
    display: flex; align-items: center; padding: 8px 6px;
    border-radius: 6px; transition: background 0.2s;
  }}
  .ap-row:hover {{ background: var(--bg-secondary); }}
  .ap-lbl {{ flex: 0 0 80px; font-size: 13px; font-weight: 700; color: var(--text-secondary); }}
  .ap-bar-wrap {{ flex: 1; padding: 0 12px; }}
  .ap-bar-bg {{ height: 8px; background: var(--bg-secondary); border-radius: 8px; overflow: hidden; }}
  .ap-bar-fill {{ height: 100%; border-radius: 8px; animation: barGrow 0.8s ease; }}
  .ap-right {{ flex: 0 0 180px; text-align: right; }}
  .ap-pct {{ font-size: 14px; font-weight: 800; color: var(--text-primary); }}
  .ap-amt {{ font-size: 11px; color: var(--text-muted); margin-left: 6px; }}
  .ap-tr {{ font-size: 10px; color: var(--text-label); display: block; margin-top: 2px; }}
</style>
<div class="csm-dash">
  <div class="alloc-duo">
    <div class="alloc-panel">
      <div class="ap-header"><h3 class="ap-title">By Business Group</h3><span class="ap-sub">% of {TARGET_QUARTER} target</span></div>
      <div id="alloc-bg-rows"></div>
    </div>
    <div class="alloc-panel">
      <div class="ap-header"><h3 class="ap-title">By Business Unit</h3><span class="ap-sub">% of {TARGET_QUARTER} target</span></div>
      <div id="alloc-bu-rows"></div>
    </div>
  </div>
</div>
<script>
  (function() {{
    function renderBars(data, containerId) {{
      const ct = document.getElementById(containerId);
      ct.innerHTML = data.map((item, i) => `
        <div class="ap-row" style="animation:slideInRight 0.3s ease ${{i*0.06}}s both">
          <div class="ap-lbl">${{item.label}}</div>
          <div class="ap-bar-wrap">
            <div class="ap-bar-bg"><div class="ap-bar-fill" style="width:${{item.pct}}%;background:${{item.color}}"></div></div>
          </div>
          <div class="ap-right">
            <span class="ap-pct">${{item.pct}}%</span><span class="ap-amt">${{item.amt}}</span>
            <span class="ap-tr">TR: ${{item.tr.toFixed(2)}}%</span>
          </div>
        </div>
      `).join('');
    }}
    renderBars({_bg_payload}, 'alloc-bg-rows');
    renderBars({_bu_payload}, 'alloc-bu-rows');
  }})();
</script>
""")


In [0]:
# Target Allocation — By Client Motion

_alloc_df = alloc_motion.copy()
_total = _alloc_df["target_rev"].sum()
_alloc_df["pct"] = np.where(_total > 0, _alloc_df["target_rev"] / _total * 100, 0)
_color_map = MOTION_COLORS

_items = []
for i, row in _alloc_df.iterrows():
    lbl = str(row["account_motion"])
    _items.append({
        "label": lbl,
        "pct": round(row["pct"], 1),
        "amt": format_usd(row["target_rev"]),
        "l4q_tr": round(row.get("l4q_tr", 0), 4),
        "color": _color_map.get(lbl, PALETTE[i % len(PALETTE)]),
    })

_payload = json.dumps(_items)

displayHTML(f"""
<style>
  .bar-card {{
    width: 100%; background: var(--bg-primary);
    border: 1px solid var(--border-light);
    border-radius: var(--radius-md);
    padding: 24px;
    box-shadow: var(--shadow-sm);
    font-family: var(--font-sans);
    transition: box-shadow 0.3s;
    animation: fadeInUp 0.4s ease-out;
  }}
  .bar-card:hover {{ box-shadow: var(--shadow-md); }}
  .bar-header {{ margin-bottom: 18px; padding-bottom: 12px; border-bottom: 1px solid var(--border-light); }}
  .bar-title {{ font-size: 16px; font-weight: 800; color: var(--text-primary); margin: 0 0 3px; }}
  .bar-sub {{ font-size: 10px; font-weight: 700; color: var(--text-label); text-transform: uppercase; letter-spacing: 0.08em; }}
  .bar-list {{ display: flex; flex-direction: column; gap: 2px; }}
  .bar-row {{
    display: flex; align-items: center; padding: 10px 8px;
    border-radius: var(--radius-sm);
    transition: background 0.2s;
  }}
  .bar-row:hover {{ background: var(--bg-secondary); }}
  .bar-label {{
    flex: 0 0 160px; font-size: 13px; font-weight: 600; color: var(--text-secondary);
    white-space: nowrap; overflow: hidden; text-overflow: ellipsis;
  }}
  .bar-track {{ flex: 1; padding: 0 16px; }}
  .bar-bg {{ width: 100%; height: 6px; background: var(--bg-secondary); border-radius: 10px; overflow: hidden; }}
  .bar-fill {{ height: 100%; border-radius: 10px; animation: barGrow 0.8s ease-out; }}
  .bar-meta {{ flex: 0 0 200px; text-align: right; font-variant-numeric: tabular-nums; display: flex; align-items: center; justify-content: flex-end; gap: 8px; }}
  .bar-pct {{ font-size: 13px; font-weight: 700; color: var(--text-primary); }}
  .bar-amt {{ font-size: 11px; font-weight: 500; color: var(--text-muted); }}
  .bar-tr {{ font-size: 10px; font-weight: 600; color: var(--text-label); white-space: nowrap; }}
</style>
<div class="csm-dash">
  <div class="bar-card">
    <div class="bar-header">
      <h3 class="bar-title">By Client Motion</h3>
      <span class="bar-sub">% of {TARGET_QUARTER} target revenue</span>
    </div>
    <div id="bar-list-account_motion" class="bar-list"></div>
  </div>
</div>
<script>
  (function() {{
    const data = {_payload};
    const ct = document.getElementById('bar-list-account_motion');
    ct.innerHTML = data.map((item, i) => {{
      let rowHtml = `
        <div class="bar-row" style="animation:slideInRight 0.3s ease ${{i*0.05}}s both">
          <div class="bar-label" title="${{item.label}}">${{item.label}}</div>
          <div class="bar-track">
            <div class="bar-bg"><div class="bar-fill" style="width:${{item.pct}}%;background:${{item.color}};"></div></div>
          </div>
          <div class="bar-meta">
            <span class="bar-pct">${{item.pct}}%</span>
            <span class="bar-amt">${{item.amt}}</span>`;
      
        const trVal = item.l4q_tr !== undefined ? (item.l4q_tr * 100).toFixed(2) + '%' : '';
        rowHtml += `<span class="bar-tr">TR: ${trVal}</span>`;
        
      rowHtml += `</div></div>`;
      return rowHtml;
    }}).join('');
  }})();
</script>
""")


In [0]:
# Target Allocation — By CSM

_alloc_df = alloc_csm.head(20).copy()
_total = _alloc_df["target_rev"].sum()
_alloc_df["pct"] = np.where(_total > 0, _alloc_df["target_rev"] / _total * 100, 0)
_color_map = {}

_items = []
for i, row in _alloc_df.iterrows():
    lbl = str(row["account_manager"])
    _items.append({
        "label": lbl,
        "pct": round(row["pct"], 1),
        "amt": format_usd(row["target_rev"]),
        "l4q_tr": round(row.get("l4q_tr", 0), 4),
        "color": _color_map.get(lbl, PALETTE[i % len(PALETTE)]),
    })

_payload = json.dumps(_items)

displayHTML(f"""
<style>
  .bar-card {{
    width: 100%; background: var(--bg-primary);
    border: 1px solid var(--border-light);
    border-radius: var(--radius-md);
    padding: 24px;
    box-shadow: var(--shadow-sm);
    font-family: var(--font-sans);
    transition: box-shadow 0.3s;
    animation: fadeInUp 0.4s ease-out;
  }}
  .bar-card:hover {{ box-shadow: var(--shadow-md); }}
  .bar-header {{ margin-bottom: 18px; padding-bottom: 12px; border-bottom: 1px solid var(--border-light); }}
  .bar-title {{ font-size: 16px; font-weight: 800; color: var(--text-primary); margin: 0 0 3px; }}
  .bar-sub {{ font-size: 10px; font-weight: 700; color: var(--text-label); text-transform: uppercase; letter-spacing: 0.08em; }}
  .bar-list {{ display: flex; flex-direction: column; gap: 2px; }}
  .bar-row {{
    display: flex; align-items: center; padding: 10px 8px;
    border-radius: var(--radius-sm);
    transition: background 0.2s;
  }}
  .bar-row:hover {{ background: var(--bg-secondary); }}
  .bar-label {{
    flex: 0 0 160px; font-size: 13px; font-weight: 600; color: var(--text-secondary);
    white-space: nowrap; overflow: hidden; text-overflow: ellipsis;
  }}
  .bar-track {{ flex: 1; padding: 0 16px; }}
  .bar-bg {{ width: 100%; height: 6px; background: var(--bg-secondary); border-radius: 10px; overflow: hidden; }}
  .bar-fill {{ height: 100%; border-radius: 10px; animation: barGrow 0.8s ease-out; }}
  .bar-meta {{ flex: 0 0 200px; text-align: right; font-variant-numeric: tabular-nums; display: flex; align-items: center; justify-content: flex-end; gap: 8px; }}
  .bar-pct {{ font-size: 13px; font-weight: 700; color: var(--text-primary); }}
  .bar-amt {{ font-size: 11px; font-weight: 500; color: var(--text-muted); }}
  .bar-tr {{ font-size: 10px; font-weight: 600; color: var(--text-label); white-space: nowrap; }}
</style>
<div class="csm-dash">
  <div class="bar-card">
    <div class="bar-header">
      <h3 class="bar-title">By CSM</h3>
      <span class="bar-sub">% of {TARGET_QUARTER} target revenue — Top 20 managers</span>
    </div>
    <div id="bar-list-account_manager" class="bar-list"></div>
  </div>
</div>
<script>
  (function() {{
    const data = {_payload};
    const ct = document.getElementById('bar-list-account_manager');
    ct.innerHTML = data.map((item, i) => {{
      let rowHtml = `
        <div class="bar-row" style="animation:slideInRight 0.3s ease ${{i*0.05}}s both">
          <div class="bar-label" title="${{item.label}}">${{item.label}}</div>
          <div class="bar-track">
            <div class="bar-bg"><div class="bar-fill" style="width:${{item.pct}}%;background:${{item.color}};"></div></div>
          </div>
          <div class="bar-meta">
            <span class="bar-pct">${{item.pct}}%</span>
            <span class="bar-amt">${{item.amt}}</span>`;
      
        const trVal = item.l4q_tr !== undefined ? (item.l4q_tr * 100).toFixed(2) + '%' : '';
        rowHtml += `<span class="bar-tr">TR: ${trVal}</span>`;
        
      rowHtml += `</div></div>`;
      return rowHtml;
    }}).join('');
  }})();
</script>
""")


In [0]:
# ═══════════════════════════════════════════════════════════════════════
# BU x Motion Cross-Tab — Grid Cards
# ═══════════════════════════════════════════════════════════════════════

cross = alloc_bu_mot.copy()
cross["label"] = cross["business_unit"] + " / " + cross["account_motion"]
cross["pct"]   = np.where(total_target_rev > 0, cross["target_rev"] / total_target_rev * 100, 0)

items = []
for i, row in cross.iterrows():
    items.append({
        "label": row["label"],
        "pct": round(row["pct"], 1),
        "amt": format_usd(row["target_rev"]),
        "tr": fmt_pct(row["l4q_tr"]),
        "color": PALETTE[i % len(PALETTE)],
    })

payload = json.dumps(items)

displayHTML(f"""
<style>
  .xtab-wrap {{
    background: var(--bg-primary); border: 1px solid var(--border-light);
    border-radius: var(--radius-md); padding: 24px;
    box-shadow: var(--shadow-sm); margin: 12px 0;
    font-family: var(--font-sans);
    animation: fadeInUp 0.4s ease-out;
  }}
  .xtab-header {{ margin-bottom: 18px; padding-bottom: 10px; border-bottom: 1px solid var(--border-light); }}
  .xtab-title {{ font-size: 16px; font-weight: 800; color: var(--text-primary); margin: 0 0 3px; }}
  .xtab-sub {{ font-size: 10px; font-weight: 700; color: var(--text-label); text-transform: uppercase; letter-spacing: 0.08em; }}
  .xtab-grid {{
    display: grid;
    grid-template-columns: repeat(auto-fill, minmax(200px, 1fr));
    gap: 12px;
  }}
  .xtab-card {{
    background: var(--bg-secondary); border: 1px solid var(--border-light);
    border-radius: 12px; padding: 18px; text-align: center;
    transition: all 0.25s; position: relative; overflow: hidden;
  }}
  .xtab-card:hover {{ transform: translateY(-3px); box-shadow: var(--shadow-md); border-color: var(--border-medium); }}
  .xtab-card::before {{
    content: '';
    position: absolute; top: 0; left: 0;
    width: 100%; height: 3px;
  }}
  .xtab-lbl {{ font-size: 11px; font-weight: 700; color: var(--text-muted); text-transform: uppercase; letter-spacing: 0.04em; margin-bottom: 8px; }}
  .xtab-val {{ font-size: 22px; font-weight: 800; color: var(--text-primary); line-height: 1; }}
  .xtab-meta {{ font-size: 11px; font-weight: 600; color: var(--accent-blue); margin-top: 8px; }}
</style>
<div class="csm-dash">
  <div class="xtab-wrap">
    <div class="xtab-header">
      <h3 class="xtab-title">BU x Client Motion Cross-Tab</h3>
      <span class="xtab-sub">Target Revenue for {TARGET_QUARTER}</span>
    </div>
    <div id="xtab-grid-v2" class="xtab-grid"></div>
  </div>
</div>
<script>
  (function() {{
    const data = {payload};
    document.getElementById('xtab-grid-v2').innerHTML = data.map((item, i) => `
      <div class="xtab-card" style="animation:fadeInUp 0.3s ease ${{i*0.08}}s both">
        <div style="position:absolute;top:0;left:0;width:100%;height:3px;background:${{item.color}}"></div>
        <div class="xtab-lbl">${{item.label}}</div>
        <div class="xtab-val">${{item.amt}}</div>
        <div class="xtab-meta">${{item.pct}}% of total &middot; TR: ${{item.tr}}</div>
      </div>
    `).join('');
  }})();
</script>
""")


In [0]:
# ═══════════════════════════════════════════════════════════════════════
# CSM x Motion — Detail Table
# ═══════════════════════════════════════════════════════════════════════

csm_m = alloc_csm_mot.copy()
csm_m["target_rev_fmt"]  = csm_m["target_rev"].apply(format_usd)
csm_m["take_rate_fmt"]   = csm_m["l4q_tr"].apply(fmt_pct)
csm_m["contrib_pct"]     = (csm_m["avg_contrib"] * 100).round(1)

rows = []
for _, r in csm_m.sort_values("target_rev", ascending=False).iterrows():
    rows.append({
        "csm": r["account_manager"],
        "motion": r["account_motion"],
        "rev": r["target_rev_fmt"],
        "pct": f'{r["contrib_pct"]}%',
        "tr": r["take_rate_fmt"],
    })

payload = json.dumps(rows)

displayHTML(f"""
<style>
  .dtbl-wrap {{
    background: var(--bg-primary); border: 1px solid var(--border-light);
    border-radius: var(--radius-md); padding: 24px;
    box-shadow: var(--shadow-sm); margin: 12px 0;
    font-family: var(--font-sans); max-height: 560px; overflow-y: auto;
    animation: fadeInUp 0.4s ease-out;
  }}
  .dtbl-header {{ margin-bottom: 14px; padding-bottom: 10px; border-bottom: 1px solid var(--border-light); }}
  .dtbl-title {{ font-size: 16px; font-weight: 800; color: var(--text-primary); margin: 0 0 3px; }}
  .dtbl-sub {{ font-size: 10px; font-weight: 700; color: var(--text-label); text-transform: uppercase; }}
  table.dtbl {{ width: 100%; border-collapse: collapse; font-size: 13px; }}
  table.dtbl th {{
    text-align: left; font-size: 10px; font-weight: 700; color: var(--text-label);
    text-transform: uppercase; letter-spacing: 0.08em;
    padding: 10px 12px; border-bottom: 2px solid var(--border-light);
    position: sticky; top: 0; background: var(--bg-primary);
  }}
  table.dtbl td {{ padding: 10px 12px; border-bottom: 1px solid var(--border-light); color: var(--text-primary); }}
  table.dtbl tr:hover {{ background: var(--bg-secondary); }}
  table.dtbl .td-r {{ text-align: right; font-weight: 600; font-variant-numeric: tabular-nums; }}
</style>
<div class="csm-dash">
  <div class="dtbl-wrap">
    <div class="dtbl-header">
      <h3 class="dtbl-title">CSM x Client Motion Detail</h3>
      <span class="dtbl-sub">Sorted by target revenue &bull; {TARGET_QUARTER}</span>
    </div>
    <table class="dtbl">
      <thead><tr><th>CSM</th><th>Motion</th><th style="text-align:right">Target Revenue</th><th style="text-align:right">Contribution</th><th style="text-align:right">L4Q Take Rate</th></tr></thead>
      <tbody id="csm-mot-body"></tbody>
    </table>
  </div>
</div>
<script>
  (function() {{
    const data = {payload};
    document.getElementById('csm-mot-body').innerHTML = data.map(r => `
      <tr>
        <td>${{r.csm}}</td><td>${{r.motion}}</td>
        <td class="td-r">${{r.rev}}</td><td class="td-r">${{r.pct}}</td><td class="td-r">${{r.tr}}</td>
      </tr>
    `).join('');
  }})();
</script>
""")


In [0]:
displayHTML(f"""
<style>
  .sec-banner {{
    display: flex;
    align-items: center;
    gap: 16px;
    padding: 20px 0;
    margin: 24px 0 8px;
    border-bottom: 2px solid #1A1A18;
    animation: fadeInUp 0.4s ease-out;
  }}
  .sec-dot {{
    width: 10px; height: 10px;
    border-radius: 50%;
    background: #3B82F6;
  }}
  .sec-title {{
    font-size: 20px;
    font-weight: 800;
    color: #1A1A18;
    letter-spacing: -0.01em;
    font-family: var(--font-sans);
  }}
  .sec-subtitle {{
    font-size: 12px;
    font-weight: 600;
    color: #92918C;
    margin-left: auto;
  }}
</style>
<div class="csm-dash">
  <div class="sec-banner">
    <div class="sec-dot"></div>
    <div class="sec-title">CSM Capacity Analysis</div>
    <div class="sec-subtitle">Concentration risk, growth difficulty & capacity scores</div>
  </div>
</div>
""")

In [0]:
# ═══════════════════════════════════════════════════════════════════════
# CSM Capacity — Summary Chips + Scorecard Table
# ═══════════════════════════════════════════════════════════════════════

_stretch_count = int((csm_capacity["growth_difficulty"] == "Stretch").sum())
_high_conc     = int((csm_capacity["concentration_flag"] == "High Risk").sum())
_healthy_count = int((csm_capacity["concentration_flag"] == "Healthy").sum())
_avg_score     = int(csm_capacity["capacity_score"].mean())
_total_csms    = len(csm_capacity)

cap_rows = []
for _, r in csm_capacity.sort_values("capacity_score", ascending=True).iterrows():
    cap_rows.append({
        "csm": r["account_manager"],
        "clients": int(r["client_count"]),
        "target": format_usd(r["target_rev"]),
        "avg_client": format_usd(r["avg_client_target"]),
        "conc": round(r["top3_concentration"] * 100),
        "conc_flag": r["concentration_flag"],
        "req_growth": round(r["required_growth"] * 100, 1),
        "growth_flag": r["growth_difficulty"],
        "score": int(r["capacity_score"]),
    })

cap_payload = json.dumps(cap_rows)

displayHTML(f"""
<style>
  .cap-chips {{
    display: flex; gap: 12px; margin: 12px 0;
    animation: fadeInUp 0.4s ease-out;
  }}
  .cap-chip {{
    flex: 1; background: var(--bg-primary);
    border: 1px solid var(--border-light);
    border-radius: 12px; padding: 16px 20px; text-align: center;
    transition: all 0.25s;
  }}
  .cap-chip:hover {{ box-shadow: var(--shadow-md); transform: translateY(-2px); }}
  .cap-chip-val {{ font-size: 28px; font-weight: 800; line-height: 1; }}
  .cap-chip-label {{ font-size: 10px; font-weight: 700; color: var(--text-label); text-transform: uppercase; letter-spacing: 0.08em; margin-top: 6px; }}

  .cap-tbl-wrap {{
    background: var(--bg-primary); border: 1px solid var(--border-light);
    border-radius: var(--radius-md); padding: 24px;
    box-shadow: var(--shadow-sm); margin: 12px 0;
    font-family: var(--font-sans); max-height: 600px; overflow-y: auto;
    animation: fadeInUp 0.4s ease-out 0.1s both;
  }}
  .cap-tbl-header {{ margin-bottom: 14px; padding-bottom: 10px; border-bottom: 1px solid var(--border-light); }}
  .cap-tbl-title {{ font-size: 16px; font-weight: 800; color: var(--text-primary); margin: 0 0 3px; }}
  .cap-tbl-sub {{ font-size: 10px; font-weight: 700; color: var(--text-label); text-transform: uppercase; }}
  table.cap-tbl {{ width: 100%; border-collapse: collapse; font-size: 12px; }}
  table.cap-tbl th {{
    text-align: left; font-size: 10px; font-weight: 700; color: var(--text-label);
    text-transform: uppercase; letter-spacing: 0.06em;
    padding: 8px 10px; border-bottom: 2px solid var(--border-light);
    position: sticky; top: 0; background: var(--bg-primary);
  }}
  table.cap-tbl td {{ padding: 9px 10px; border-bottom: 1px solid var(--border-light); }}
  table.cap-tbl tr:hover {{ background: var(--bg-secondary); }}
  .flag {{
    display: inline-block; padding: 2px 8px; border-radius: 4px;
    font-size: 10px; font-weight: 700; white-space: nowrap;
  }}
  .flag-healthy {{ background: #ecfdf5; color: #047857; }}
  .flag-moderate {{ background: #fffbeb; color: #b45309; }}
  .flag-risk {{ background: #fef2f2; color: #b91c1c; }}
  .flag-conservative {{ background: #ecfdf5; color: #047857; }}
  .flag-achievable {{ background: #eff6ff; color: #1d4ed8; }}
  .flag-stretch {{ background: #fef2f2; color: #b91c1c; }}
  .score-wrap {{ display: flex; align-items: center; gap: 6px; }}
  .score-bar {{ height: 6px; border-radius: 4px; }}
  .score-num {{ font-weight: 800; font-size: 12px; }}
</style>
<div class="csm-dash">
  <div class="cap-chips">
    <div class="cap-chip">
      <div class="cap-chip-val" style="color:var(--text-primary);">{_total_csms}</div>
      <div class="cap-chip-label">Total CSMs</div>
    </div>
    <div class="cap-chip">
      <div class="cap-chip-val" style="color:var(--accent-blue);">{_avg_score}</div>
      <div class="cap-chip-label">Avg Capacity Score</div>
    </div>
    <div class="cap-chip">
      <div class="cap-chip-val" style="color:var(--accent-green);">{_healthy_count}</div>
      <div class="cap-chip-label">Healthy Concentration</div>
    </div>
    <div class="cap-chip">
      <div class="cap-chip-val" style="color:var(--accent-red);">{_high_conc}</div>
      <div class="cap-chip-label">High Concentration Risk</div>
    </div>
    <div class="cap-chip">
      <div class="cap-chip-val" style="color:var(--accent-amber);">{_stretch_count}</div>
      <div class="cap-chip-label">Stretch Targets</div>
    </div>
  </div>

  <div class="cap-tbl-wrap">
    <div class="cap-tbl-header">
      <h3 class="cap-tbl-title">CSM Capacity Scorecard</h3>
      <span class="cap-tbl-sub">Sorted by score (lowest first = highest risk)</span>
    </div>
    <table class="cap-tbl">
      <thead><tr>
        <th>CSM</th><th style="text-align:center">Clients</th><th style="text-align:right">Target</th>
        <th style="text-align:right">Avg/Client</th><th style="text-align:center">Top-3 Conc.</th>
        <th style="text-align:center">Flag</th><th style="text-align:right">Req. Growth</th>
        <th style="text-align:center">Difficulty</th><th style="text-align:center">Score</th>
      </tr></thead>
      <tbody id="cap-tbl-body"></tbody>
    </table>
  </div>
</div>
<script>
  (function() {{
    const data = {cap_payload};
    const fc = {{"Healthy":"flag-healthy","Moderate":"flag-moderate","High Risk":"flag-risk",
                 "Conservative":"flag-conservative","Achievable":"flag-achievable","Stretch":"flag-stretch"}};
    const sc = s => s >= 70 ? '#047857' : s >= 40 ? '#b45309' : '#b91c1c';
    document.getElementById('cap-tbl-body').innerHTML = data.map(r => `
      <tr>
        <td style="font-weight:600">${{r.csm}}</td>
        <td style="text-align:center">${{r.clients}}</td>
        <td style="text-align:right;font-weight:600">${{r.target}}</td>
        <td style="text-align:right;color:var(--text-muted)">${{r.avg_client}}</td>
        <td style="text-align:center;font-weight:700">${{r.conc}}%</td>
        <td style="text-align:center"><span class="flag ${{fc[r.conc_flag]||''}}">${{r.conc_flag}}</span></td>
        <td style="text-align:right;font-weight:700">${{r.req_growth > 0 ? '+' : ''}}${{r.req_growth}}%</td>
        <td style="text-align:center"><span class="flag ${{fc[r.growth_flag]||''}}">${{r.growth_flag}}</span></td>
        <td style="text-align:center">
          <div class="score-wrap">
            <div class="score-bar" style="width:${{r.score}}px;background:${{sc(r.score)}}"></div>
            <span class="score-num" style="color:${{sc(r.score)}}">${{r.score}}</span>
          </div>
        </td>
      </tr>
    `).join('');
  }})();
</script>
""")


In [0]:
displayHTML(f"""
<style>
  .sec-banner {{
    display: flex;
    align-items: center;
    gap: 16px;
    padding: 20px 0;
    margin: 24px 0 8px;
    border-bottom: 2px solid #1A1A18;
    animation: fadeInUp 0.4s ease-out;
  }}
  .sec-dot {{
    width: 10px; height: 10px;
    border-radius: 50%;
    background: #3B82F6;
  }}
  .sec-title {{
    font-size: 20px;
    font-weight: 800;
    color: #1A1A18;
    letter-spacing: -0.01em;
    font-family: var(--font-sans);
  }}
  .sec-subtitle {{
    font-size: 12px;
    font-weight: 600;
    color: #92918C;
    margin-left: auto;
  }}
</style>
<div class="csm-dash">
  <div class="sec-banner">
    <div class="sec-dot"></div>
    <div class="sec-title">Profit Guardrails</div>
    <div class="sec-subtitle">Take Rate vs Historical Avg — {TARGET_QUARTER}</div>
  </div>
</div>
""")

In [0]:
# ═══════════════════════════════════════════════════════════════════════
# Guardrail Summary + Revamped Risk Clients Table
# ═══════════════════════════════════════════════════════════════════════

safe_pct = round(safe_count / total_clients * 100, 1) if total_clients else 0
risk_pct = round(risk_count / total_clients * 100, 1) if total_clients else 0

risk_df = alloc_client[alloc_client["guardrail"] == "Risk"].sort_values("tr_delta").head(25)
risk_rows = []
for idx, (_, r) in enumerate(risk_df.iterrows()):
    risk_rows.append({
        "rank": idx + 1,
        "name": r["company_name"],
        "ref": r.get("business_ref_code", ""),
        "bg": r["business_group"],
        "bu": r["business_unit"],
        "motion": r["account_motion"],
        "csm": r["account_manager"],
        "rev": format_usd(r["target_revenue"]),
        "rev_raw": float(r["target_revenue"]),
        "tgt_tr": round(r["target_take_rate"] * 100, 2),
        "hist_tr": round(r["hist_avg_tr"] * 100, 2),
        "delta": round(r["tr_delta"] * 100, 2),
    })
risk_payload = json.dumps(risk_rows)

# Compute max revenue for bar width scaling
_max_risk_rev = risk_df["target_revenue"].max() if len(risk_df) > 0 else 1

displayHTML(f"""
<style>
  .guard-row {{
    display: grid;
    grid-template-columns: 1fr 1fr 1fr;
    gap: 16px;
    margin: 12px 0;
    animation: fadeInUp 0.4s ease-out;
  }}
  .guard-card {{
    background: var(--bg-primary); border: 1px solid var(--border-light);
    border-radius: var(--radius-md); padding: 28px; text-align: center;
    position: relative; overflow: hidden; transition: all 0.25s;
  }}
  .guard-card:hover {{ box-shadow: var(--shadow-md); transform: translateY(-2px); }}
  .guard-val {{ font-size: 52px; font-weight: 900; line-height: 1; }}
  .guard-label {{ font-size: 11px; font-weight: 700; color: var(--text-label); text-transform: uppercase; letter-spacing: 0.1em; margin-bottom: 12px; }}
  .guard-sub {{ font-size: 13px; font-weight: 600; margin-top: 8px; }}

  /* Revamped risk table */
  .risk-panel {{
    background: var(--bg-primary); border: 1px solid var(--border-light);
    border-radius: var(--radius-lg); padding: 0;
    box-shadow: var(--shadow-sm); margin: 16px 0;
    font-family: var(--font-sans);
    overflow: hidden;
    animation: fadeInUp 0.4s ease-out 0.1s both;
  }}
  .risk-panel-header {{
    padding: 20px 28px;
    background: linear-gradient(135deg, #fef2f2 0%, #fff5f5 100%);
    border-bottom: 1px solid #fecaca;
    display: flex; justify-content: space-between; align-items: center;
  }}
  .risk-panel-title {{
    font-size: 16px; font-weight: 800; color: #991b1b;
    display: flex; align-items: center; gap: 10px;
  }}
  .risk-panel-title::before {{
    content: '';
    width: 8px; height: 8px; border-radius: 50%;
    background: #ef4444;
    animation: pulseGlow 2s infinite;
  }}
  .risk-panel-count {{
    font-size: 12px; font-weight: 700; color: #b91c1c;
    background: #fee2e2; padding: 4px 14px; border-radius: 20px;
  }}
  .risk-tbl-scroll {{
    max-height: 520px; overflow-y: auto;
    scrollbar-width: thin;
    scrollbar-color: #ddd transparent;
  }}
  table.risk-tbl2 {{ width: 100%; border-collapse: collapse; font-size: 12px; }}
  table.risk-tbl2 thead {{ position: sticky; top: 0; z-index: 2; }}
  table.risk-tbl2 th {{
    text-align: left; font-size: 10px; font-weight: 700; color: var(--text-label);
    text-transform: uppercase; letter-spacing: 0.06em;
    padding: 12px 14px; background: #fafaf8;
    border-bottom: 2px solid var(--border-light);
  }}
  table.risk-tbl2 td {{
    padding: 12px 14px;
    border-bottom: 1px solid var(--border-light);
    vertical-align: middle;
  }}
  table.risk-tbl2 tbody tr {{
    transition: all 0.15s ease;
  }}
  table.risk-tbl2 tbody tr:nth-child(even) {{ background: var(--bg-secondary); }}
  table.risk-tbl2 tbody tr:hover {{ background: #fef2f2; }}
  .risk-rank {{
    width: 24px; height: 24px; border-radius: 6px;
    background: #fee2e2; color: #b91c1c;
    font-size: 10px; font-weight: 800;
    display: flex; align-items: center; justify-content: center;
  }}
  .risk-name {{ font-weight: 700; color: var(--text-primary); }}
  .risk-name-sub {{ font-size: 10px; color: var(--text-muted); margin-top: 2px; }}
  .risk-pill {{
    display: inline-block; padding: 3px 8px; border-radius: 6px;
    font-size: 10px; font-weight: 700; white-space: nowrap;
  }}
  .risk-pill-bg {{ background: var(--bg-secondary); color: var(--text-secondary); }}
  .risk-tr-bar {{
    display: flex; align-items: center; gap: 6px;
  }}
  .risk-tr-track {{
    flex: 1; height: 6px; background: #fee2e2; border-radius: 4px;
    overflow: hidden; position: relative; min-width: 50px;
  }}
  .risk-tr-fill {{
    height: 100%; border-radius: 4px;
    background: linear-gradient(90deg, #ef4444, #f87171);
  }}
  .risk-tr-hist {{
    position: absolute; top: -3px; width: 2px; height: 12px;
    background: #10B981; border-radius: 1px;
  }}
  .risk-delta-badge {{
    display: inline-flex; align-items: center; gap: 3px;
    padding: 3px 10px; border-radius: 6px;
    font-size: 11px; font-weight: 800;
    background: #fee2e2; color: #b91c1c;
  }}
  .risk-rev-bar {{
    height: 4px; border-radius: 2px;
    background: linear-gradient(90deg, #fca5a5, #fecaca);
    margin-top: 4px;
  }}
</style>
<div class="csm-dash">
  <div class="guard-row">
    <div class="guard-card">
      <div style="position:absolute;top:0;left:0;width:100%;height:3px;background:var(--accent-green);"></div>
      <div class="guard-label">Safe Clients</div>
      <div class="guard-val" style="color:var(--accent-green);">{safe_count}</div>
      <div class="guard-sub" style="color:var(--accent-green);">{safe_pct}% of portfolio</div>
    </div>
    <div class="guard-card">
      <div style="position:absolute;top:0;left:0;width:100%;height:3px;background:var(--accent-red);"></div>
      <div class="guard-label">At-Risk Clients</div>
      <div class="guard-val" style="color:var(--accent-red);">{risk_count}</div>
      <div class="guard-sub" style="color:var(--accent-red);">{risk_pct}% below hist. TR</div>
    </div>
    <div class="guard-card">
      <div style="position:absolute;top:0;left:0;width:100%;height:3px;background:var(--accent-blue);"></div>
      <div class="guard-label">Historical Avg Take Rate</div>
      <div class="guard-val" style="color:var(--text-primary);">{fmt_pct(HIST_AVG_TR)}</div>
      <div class="guard-sub" style="color:var(--accent-blue);">L4Q benchmark</div>
    </div>
  </div>

  <div class="risk-panel">
    <div class="risk-panel-header">
      <div class="risk-panel-title">At-Risk Client Detail</div>
      <div class="risk-panel-count">{risk_count} clients below historical TR</div>
    </div>
    <div class="risk-tbl-scroll">
      <table class="risk-tbl2">
        <thead><tr>
          <th style="width:30px">#</th>
          <th>Company</th>
          <th>Group</th>
          <th>CSM</th>
          <th style="text-align:right">Target Revenue</th>
          <th style="text-align:center;min-width:140px">Take Rate Comparison</th>
          <th style="text-align:center">Delta</th>
        </tr></thead>
        <tbody id="risk-tbl2-body"></tbody>
      </table>
    </div>
  </div>
</div>
<script>
  (function() {{
    const data = {risk_payload};
    const maxRev = Math.max(...data.map(r => r.rev_raw), 1);
    document.getElementById('risk-tbl2-body').innerHTML = data.map((r, i) => {{
      const barW = Math.round(r.rev_raw / maxRev * 100);
      const trMax = Math.max(r.tgt_tr, r.hist_tr, 1);
      const fillW = Math.round(r.tgt_tr / trMax * 100);
      const histPos = Math.round(r.hist_tr / trMax * 100);
      return `
        <tr style="animation:fadeInUp 0.25s ease ${{i*0.03}}s both">
          <td><div class="risk-rank">${{r.rank}}</div></td>
          <td>
            <div class="risk-name">${{r.name}}</div>
            <div class="risk-name-sub">${{r.bu}} &middot; ${{r.motion}}</div>
          </td>
          <td><span class="risk-pill risk-pill-bg">${{r.bg}}</span></td>
          <td style="font-weight:600;color:var(--text-secondary)">${{r.csm}}</td>
          <td style="text-align:right">
            <div style="font-weight:700">${{r.rev}}</div>
            <div class="risk-rev-bar" style="width:${{barW}}%"></div>
          </td>
          <td>
            <div class="risk-tr-bar">
              <span style="font-size:11px;font-weight:700;color:#b91c1c;min-width:42px">${{r.tgt_tr.toFixed(1)}}%</span>
              <div class="risk-tr-track">
                <div class="risk-tr-fill" style="width:${{fillW}}%"></div>
                <div class="risk-tr-hist" style="left:${{histPos}}%" title="Historical: ${{r.hist_tr}}%"></div>
              </div>
              <span style="font-size:10px;color:#047857;min-width:42px;text-align:right">${{r.hist_tr.toFixed(1)}}%</span>
            </div>
          </td>
          <td style="text-align:center">
            <div class="risk-delta-badge">&#9660; ${{Math.abs(r.delta).toFixed(2)}}pp</div>
          </td>
        </tr>`;
    }}).join('');
  }})();
</script>
""")


In [0]:
displayHTML(f"""
<style>
  .sec-banner {{
    display: flex;
    align-items: center;
    gap: 16px;
    padding: 20px 0;
    margin: 24px 0 8px;
    border-bottom: 2px solid #1A1A18;
    animation: fadeInUp 0.4s ease-out;
  }}
  .sec-dot {{
    width: 10px; height: 10px;
    border-radius: 50%;
    background: #3B82F6;
  }}
  .sec-title {{
    font-size: 20px;
    font-weight: 800;
    color: #1A1A18;
    letter-spacing: -0.01em;
    font-family: var(--font-sans);
  }}
  .sec-subtitle {{
    font-size: 12px;
    font-weight: 600;
    color: #92918C;
    margin-left: auto;
  }}
</style>
<div class="csm-dash">
  <div class="sec-banner">
    <div class="sec-dot"></div>
    <div class="sec-title">What-If Simulation</div>
    <div class="sec-subtitle">Adjust Take Rate & Growth Rate Override widgets above</div>
  </div>
</div>
""")

In [0]:
# ═══════════════════════════════════════════════════════════════════════
# Simulation Results — All Granularities
# ═══════════════════════════════════════════════════════════════════════

def sim_to_cards(df, label_col=None):
    items = []
    for _, r in df.iterrows():
        lbl = str(r[label_col]) if label_col else "Overall"
        delta = r.get("rev_delta", 0)
        delta_pct = r.get("rev_delta_pct", 0)
        items.append({
            "label": lbl,
            "rev": format_usd(r["target_rev"]),
            "gp": format_usd(r["sim_gp"]),
            "tr": fmt_pct(r["sim_tr"]),
            "clients": int(r["clients"]),
            "safe": int(r["safe"]),
            "risk": int(r["risk"]),
            "delta": format_usd(delta) if abs(delta) > 1 else "$0",
            "delta_pct": f"{delta_pct:+.1f}%",
            "has_delta": bool(abs(delta) > 1),
        })
    return items

all_sims = {
    "Overall": sim_to_cards(sim_overall),
    "By Business Group": sim_to_cards(sim_bg, "business_group"),
    "By Business Unit": sim_to_cards(sim_bu, "business_unit"),
    "By Client Motion": sim_to_cards(sim_motion, "account_motion"),
    "By CSM (Top 15)": sim_to_cards(sim_csm.head(15), "account_manager"),
}

sim_payload = json.dumps(all_sims)
_tr_ov = f"{OVERRIDE_TR*100:.2f}%" if USE_TR_OVERRIDE else "None"
_gr_ov = f"{float(_gr_override_raw):.2f}%" if GROWTH_OVERRIDE else "None"
_any_override = USE_TR_OVERRIDE or GROWTH_OVERRIDE

displayHTML(f"""
<style>
  .sim-wrap {{
    font-family: var(--font-sans);
    animation: fadeInUp 0.4s ease-out;
  }}
  .sim-badges {{
    display: flex; gap: 10px; margin-bottom: 16px; flex-wrap: wrap;
  }}
  .sim-badge-item {{
    display: inline-flex; align-items: center; gap: 6px;
    padding: 6px 16px; border-radius: 8px;
    font-size: 12px; font-weight: 700;
    background: {'#fef2f2' if _any_override else 'var(--bg-secondary)'};
    color: {'#b91c1c' if _any_override else 'var(--text-muted)'};
    border: 1px solid {'#fca5a5' if _any_override else 'var(--border-light)'};
  }}
  .sim-badge-dot {{
    width: 6px; height: 6px; border-radius: 50%;
    background: {'#ef4444' if _any_override else 'var(--text-label)'};
  }}
  .sim-group-label {{
    font-size: 14px; font-weight: 800; color: var(--text-primary);
    margin: 20px 0 10px; padding-left: 4px;
  }}
  .sim-grid {{
    display: grid;
    grid-template-columns: repeat(auto-fill, minmax(220px, 1fr));
    gap: 12px; margin-bottom: 8px;
  }}
  .sim-card {{
    background: var(--bg-primary); border: 1px solid var(--border-light);
    border-radius: 12px; padding: 16px; transition: all 0.25s;
  }}
  .sim-card:hover {{ box-shadow: var(--shadow-md); transform: translateY(-2px); }}
  .sim-card-lbl {{ font-size: 10px; font-weight: 700; color: var(--text-label); text-transform: uppercase; letter-spacing: 0.05em; margin-bottom: 8px; }}
  .sim-card-rev {{ font-size: 20px; font-weight: 800; color: var(--text-primary); }}
  .sim-card-gp {{ font-size: 15px; font-weight: 700; color: var(--accent-green); margin-top: 4px; }}
  .sim-card-delta {{ font-size: 10px; font-weight: 600; color: var(--accent-blue); margin-top: 3px; }}
  .sim-card-foot {{ font-size: 10px; color: var(--text-muted); margin-top: 8px; display: flex; align-items: center; gap: 6px; flex-wrap: wrap; }}
  .sim-pill {{
    display: inline-block; padding: 1px 6px; border-radius: 4px; font-size: 9px; font-weight: 700;
  }}
  .sim-pill-safe {{ background: #ecfdf5; color: #047857; }}
  .sim-pill-risk {{ background: #fef2f2; color: #b91c1c; }}
</style>
<div class="csm-dash">
  <div class="sim-wrap">
    <div class="sim-badges">
      <div class="sim-badge-item"><div class="sim-badge-dot"></div>Take Rate Override: {_tr_ov}</div>
      <div class="sim-badge-item"><div class="sim-badge-dot"></div>Growth Rate Override: {_gr_ov}</div>
    </div>
    <div id="sim-render"></div>
  </div>
</div>
<script>
  (function() {{
    const data = {sim_payload};
    const ct = document.getElementById('sim-render');
    let html = '';
    for (const [group, items] of Object.entries(data)) {{
      html += `<div class="sim-group-label">${{group}}</div><div class="sim-grid">`;
      items.forEach((item, i) => {{
        const dHtml = item.has_delta ? `<div class="sim-card-delta">vs baseline: ${{item.delta}} (${{item.delta_pct}})</div>` : '';
        html += `
          <div class="sim-card" style="animation:fadeInUp 0.3s ease ${{i*0.05}}s both">
            <div class="sim-card-lbl">${{item.label}}</div>
            <div class="sim-card-rev">${{item.rev}}</div>
            <div class="sim-card-gp">GP: ${{item.gp}}</div>
            ${{dHtml}}
            <div class="sim-card-foot">
              TR: ${{item.tr}} &middot; ${{item.clients}} clients
              <span class="sim-pill sim-pill-safe">${{item.safe}} safe</span>
              <span class="sim-pill sim-pill-risk">${{item.risk}} risk</span>
            </div>
          </div>`;
      }});
      html += '</div>';
    }}
    ct.innerHTML = html;
  }})();
</script>
""")


In [0]:
displayHTML(f"""
<style>
  .sec-banner {{
    display: flex;
    align-items: center;
    gap: 16px;
    padding: 20px 0;
    margin: 24px 0 8px;
    border-bottom: 2px solid #1A1A18;
    animation: fadeInUp 0.4s ease-out;
  }}
  .sec-dot {{
    width: 10px; height: 10px;
    border-radius: 50%;
    background: #3B82F6;
  }}
  .sec-title {{
    font-size: 20px;
    font-weight: 800;
    color: #1A1A18;
    letter-spacing: -0.01em;
    font-family: var(--font-sans);
  }}
  .sec-subtitle {{
    font-size: 12px;
    font-weight: 600;
    color: #92918C;
    margin-left: auto;
  }}
</style>
<div class="csm-dash">
  <div class="sec-banner">
    <div class="sec-dot"></div>
    <div class="sec-title">Client-Level Target Detail</div>
    <div class="sec-subtitle">Full exportable table</div>
  </div>
</div>
""")

In [0]:
# ═══════════════════════════════════════════════════════════════════════
# Client-Level Target Table (native display for filtering/sorting)
# ═══════════════════════════════════════════════════════════════════════

export_df = alloc_client.copy()
export_df["target_revenue"] = export_df["target_revenue"].round(2)
export_df["target_gp"]      = export_df["target_gp"].round(2)
export_df["target_take_rate"] = (export_df["target_take_rate"] * 100).round(2)
export_df["hist_avg_tr"]    = (export_df["hist_avg_tr"] * 100).round(2)
export_df["avg_rev_g"]      = (export_df["avg_rev_g"] * 100).round(2)

display(spark.createDataFrame(export_df))


In [0]:
displayHTML(f"""
<style>
  .sec-banner {{
    display: flex;
    align-items: center;
    gap: 16px;
    padding: 20px 0;
    margin: 24px 0 8px;
    border-bottom: 2px solid #1A1A18;
    animation: fadeInUp 0.4s ease-out;
  }}
  .sec-dot {{
    width: 10px; height: 10px;
    border-radius: 50%;
    background: #3B82F6;
  }}
  .sec-title {{
    font-size: 20px;
    font-weight: 800;
    color: #1A1A18;
    letter-spacing: -0.01em;
    font-family: var(--font-sans);
  }}
  .sec-subtitle {{
    font-size: 12px;
    font-weight: 600;
    color: #92918C;
    margin-left: auto;
  }}
</style>
<div class="csm-dash">
  <div class="sec-banner">
    <div class="sec-dot"></div>
    <div class="sec-title">Export</div>
    <div class="sec-subtitle">Download final target tables</div>
  </div>
</div>
""")

In [0]:
# ═══════════════════════════════════════════════════════════════════════
# Export — save CSV to the same repo folder
# ═══════════════════════════════════════════════════════════════════════

output_name = f"csm_targets_{TARGET_QUARTER.replace('-','_')}.csv"
output_path = f"/Workspace{NB_DIR}/{output_name}"

try:
    alloc_client.to_csv(output_path, index=False)
    print(f"Exported: {output_path}")
except Exception as e:
    fallback = f"/tmp/{output_name}"
    alloc_client.to_csv(fallback, index=False)
    print(f"Exported to fallback: {fallback}")
    print(f"(Could not write to repo: {e})")

print(f"\nTotal rows: {len(alloc_client)}")
print(f"Columns: {list(alloc_client.columns)}")
